# Capstone 3 — Structured Extraction Pipeline at Scale · Practice

**Goal:** build the pipeline from the [brief](README.md#the-brief) end to end: 100 synthetic documents, a strict extraction schema, few-shot prompts, validation with retry-and-feedback, a batch run on **both** the Claude Message Batches API and the OpenAI Batch API (failures resubmitted by `custom_id`), SLA math, confidence calibration, human-review routing, accuracy by document type and field, and a cost comparison.

- Brief, requirements, acceptance criteria and rubric: [README.md](README.md)
- Estimated time: 4–6 hours
- Estimated API cost: **$0** by default (everything runs offline). With `RUN_LIVE = True`, the last section sends a 5-document sample to each real batch API: roughly **$0.01–$0.05** in total at batch prices on `claude-sonnet-5-5` and `gpt-6-luna`.
- Prerequisites: [00-prerequisites](../../../00-prerequisites/README.md) (venv from `requirements.txt`; `.env` only if you go live). Theory lives in the chapters: Claude [2 Tool use](../../../anthropic-claude/02-tool-use/README.md), [6 Prompt engineering](../../../anthropic-claude/06-prompt-engineering/README.md), [7 Message Batches](../../../anthropic-claude/07-message-batches/README.md), [9 Human in the loop](../../../anthropic-claude/09-escalation-human-in-the-loop/README.md); OpenAI [02 Structured Outputs](../../../openai-codex/02-function-calling-and-structured-outputs/README.md), [06 Evals](../../../openai-codex/06-prompt-engineering-and-evals/README.md), [07 Batch and cost](../../../openai-codex/07-batch-and-cost/README.md), [09 Human in the loop](../../../openai-codex/09-escalation-human-in-the-loop/README.md).

How the notebook runs offline:

| Piece | What it is |
|---|---|
| **Real SDK clients** | `anthropic.Anthropic` and `openai.OpenAI`, unchanged. Only their HTTP transport is swapped for `httpx2.MockTransport`, so request building, typed responses and errors are the SDKs' own code. |
| **`FakeClaudeBatches` / `FakeOpenAIBatches`** | Local stand-ins for the two batch services, with a virtual clock. They follow the documented lifecycles (`in_progress` → `ended` with four result types; `validating` → `in_progress` → `finalizing` → `completed`/`expired` with output and error files). |
| **`simulated_extraction()`** | Plays the model. It knows each synthetic document's label and applies **documented-style error profiles** (format slips, missed values, overconfident OCR mistakes, fabrication when the schema leaves no room for `null`). It is a teaching device, not a model: every number it produces is synthetic. |
| **`RUN_LIVE`** | `False` by default. Set it to `True` (and add both keys to `.env`) to run the same pipeline code against the real APIs on a small sample. |

In [ ]:
import copy
import hashlib
import json
import math
import os
import random
import re
import tempfile
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timedelta, timezone
from email.parser import BytesParser
from email.policy import HTTP
from itertools import count
from pathlib import Path
from typing import Any, Callable, Literal, Optional

import anthropic
import httpx2
import openai
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from openai.types.responses import Response
from pydantic import BaseModel, ConfigDict, ValidationError

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
RUN_LIVE = False          # opt-in: True sends a small sample to BOTH real batch APIs (needs both keys, costs cents)
LIVE_SAMPLE = 5           # documents per vendor in the live run
WORKDIR = Path(tempfile.mkdtemp(prefix="cap03_"))   # JSONL files and manifests go here, never into the repo
print("anthropic", anthropic.__version__, "| CLAUDE_MODEL =", CLAUDE_MODEL, "| OPENAI_MODEL =", OPENAI_MODEL,
      "| RUN_LIVE =", RUN_LIVE, "| scratch:", WORKDIR)

In [ ]:
# Live cells need both keys. With RUN_LIVE = False (the default) nothing is checked and everything runs offline.
if RUN_LIVE:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
    assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"

## 3.1 The synthetic corpus and its labels

The client's intake mixes six layouts: tabular invoices, invoices written as letters, OCR'd till receipts, credit notes, purchase orders and remittance advices. Some documents are missing values (no invoice number, a torn total), some contain a genuine error in the source (lines that do not add up), a few are far too long for the context window, and a few will hit transient failures. Every document carries a **label**: the answer a careful human would extract. Labels are what make calibration and accuracy-by-segment possible later. [Theory](README.md#31-the-synthetic-corpus-and-its-labels)

In [ ]:
BUYER = "Harbor & Pine Facilities"
VENDORS = ["Alder & Finch Ltd", "Brightwater Supplies", "Copperleaf Facilities", "Dunmore Electrical",
           "Elm Street Catering", "Foxglove Cleaning Co", "Granite Peak Security", "Harrow Office Furniture",
           "Ivybridge Logistics", "Juniper Pest Control", "Kestrel IT Services", "Larkspur Landscaping"]
SERVICE_ITEMS = [("Office cleaning, monthly", 30_000, 150_000), ("Window cleaning", 8_000, 40_000),
                 ("Security patrol hours", 20_000, 90_000), ("Replacement LED panels", 5_000, 60_000),
                 ("HVAC filter service", 12_000, 70_000), ("Catering, staff event", 25_000, 200_000),
                 ("Pest inspection", 6_000, 25_000), ("Courier deliveries", 2_000, 15_000),
                 ("Desk chairs", 15_000, 80_000), ("Laptop repairs", 9_000, 50_000)]
RECEIPT_ITEMS = [("coffee beans 1kg", 1_200, 3_500), ("printer paper a4", 800, 2_900), ("batteries aa x24", 900, 2_400),
                 ("cleaning spray", 300, 1_200), ("parking", 500, 2_500), ("bin liners", 400, 1_500)]
CURRENCIES = [("EUR", 50), ("USD", 20), ("GBP", 18), ("CHF", 7), ("SEK", 5)]      # CHF/SEK -> enum "other" + detail
TERMS = [("net_30", "Net 30", 30, 45), ("net_15", "Net 15", 15, 15), ("net_60", "Net 60", 60, 10),
         ("due_on_receipt", "Due on receipt", 0, 10), ("other", "2/10 Net 30", 30, 10),
         ("other", "Payable at month end", None, 5), ("unclear", "as agreed", None, 5)]
NARRATIVE_TERMS = {"Net 30": "within 30 days", "Net 15": "within 15 days", "Net 60": "within 60 days",
                   "Due on receipt": "on receipt of this letter", "2/10 Net 30": "within 30 days (2% discount if paid within 10 days)",
                   "Payable at month end": "at the end of the month", "as agreed": "as agreed"}
SYMBOL = {"EUR": "€", "GBP": "£"}
LAYOUT_DOC_TYPE = {"invoice_table": "invoice", "invoice_narrative": "invoice", "receipt_ocr": "receipt",
                   "credit_note": "credit_note", "purchase_order": "purchase_order", "remittance": "other"}
PRODUCTION_PLAN = [("invoice_table", 34), ("invoice_narrative", 16), ("receipt_ocr", 20),
                   ("credit_note", 10), ("purchase_order", 12), ("remittance", 8)]
SCORED_FIELDS = ["doc_type", "vendor_name", "document_number", "issue_date", "due_date",
                 "currency", "payment_terms", "stated_total_cents"]


@dataclass
class Doc:
    doc_id: str
    layout: str
    text: str
    truth: dict                                       # the label: what a careful human would extract
    rendered: dict = field(default_factory=dict)      # how some values are printed (used by the simulated model)
    tags: set = field(default_factory=set)            # scenario tags: oversized, long_items, source_conflict, ...


def money(cents: int) -> str:
    return f"{cents / 100:,.2f}"


def weighted(rng: random.Random, options: list[tuple]) -> tuple:
    return rng.choices(options, weights=[o[-1] for o in options])[0]


def make_doc(doc_id: str, layout: str, rng: random.Random, tags: set) -> Doc:
    vendor = rng.choice(VENDORS)
    issued = date(2026, 1, 1) + timedelta(days=rng.randrange(0, 240))
    currency = weighted(rng, CURRENCIES)[0] if layout not in ("receipt_ocr",) else rng.choice(["EUR", "GBP"])
    catalog = RECEIPT_ITEMS if layout == "receipt_ocr" else SERVICE_ITEMS
    n_items = 90 if "long_items" in tags else rng.randint(2, 4 if layout != "receipt_ocr" else 5)
    lines = []
    for _ in range(n_items):
        desc, lo, hi = rng.choice(catalog)
        lines.append({"description": desc, "amount_cents": rng.randrange(lo, hi, 5)})
    total = sum(item["amount_cents"] for item in lines)
    printed_total = total + 1_000 if "source_conflict" in tags else total     # a genuine error IN the source
    truth = {"doc_type": LAYOUT_DOC_TYPE[layout], "doc_type_detail": None, "vendor_name": vendor,
             "document_number": None, "issue_date": issued.isoformat(), "due_date": None,
             "currency": currency if currency in ("EUR", "USD", "GBP") else "other",
             "currency_detail": None if currency in ("EUR", "USD", "GBP") else currency,
             "payment_terms": None, "payment_terms_detail": None, "line_items": lines,
             "stated_total_cents": printed_total, "po_reference": None,
             "conflict_detected": "source_conflict" in tags}
    rendered = {"issue_date": issued.isoformat()}
    number = f"{rng.randint(1000, 9999)}-{rng.randint(10, 99)}"

    if layout in ("invoice_table", "invoice_narrative"):
        code, terms_text, days, _ = weighted(rng, TERMS)
        if "unclear_terms" in tags:
            code, terms_text, days = "unclear", "as agreed", None
        truth["payment_terms"] = code
        truth["payment_terms_detail"] = terms_text if code == "other" else None
        if code == "other" and days is None:          # "Payable at month end": the due date is the last day of the month
            nxt = (issued.replace(day=28) + timedelta(days=4))
            truth["due_date"] = (nxt - timedelta(days=nxt.day)).isoformat()
        elif days is not None:
            truth["due_date"] = (issued + timedelta(days=days)).isoformat()
        if "no_number" not in tags:
            truth["document_number"] = f"INV-{number}"
        if rng.random() < 0.4:
            truth["po_reference"] = f"PO-{rng.randint(10000, 99999)}"
    if layout == "invoice_table":
        head = ["INVOICE", vendor]
        if truth["document_number"]:
            head.append(f"Invoice no: {truth['document_number']}")
        head.append(f"Invoice date: {truth['issue_date']}")
        if truth["due_date"]:
            head.append(f"Due date: {truth['due_date']}")
        if truth["po_reference"]:
            head.append(f"PO ref: {truth['po_reference']}")
        head += [f"Terms: {terms_text}", f"Currency: {currency}", "", "| Description | Amount |", "|---|---|"]
        body = [f"| {li['description']} | {money(li['amount_cents'])} |" for li in lines]
        text = "\n".join(head + body + ["", f"TOTAL DUE: {money(printed_total)} {currency}"])
    elif layout == "invoice_narrative":
        long_date = f"{issued.day} {issued:%B %Y}"
        rendered["issue_date"] = long_date
        ref = f" under reference {truth['document_number']}" if truth["document_number"] else ""
        po = f" against your order {truth['po_reference']}" if truth["po_reference"] else ""
        items = ", ".join(f"{li['description'].lower()} ({currency} {money(li['amount_cents'])})" for li in lines)
        due = ""
        if truth["due_date"]:
            d = date.fromisoformat(truth["due_date"])
            due = f", that is by {d.day} {d:%B %Y}"
        text = (f"Dear Accounts Team,\n\nFollowing our work for {BUYER}, {vendor} is pleased to confirm the services "
                f"below{ref}{po}. This letter, dated {long_date}, serves as our invoice.\n\nWe provided {items}.\n\n"
                f"Taken together, the sum of {currency} {money(printed_total)} is payable {NARRATIVE_TERMS[terms_text]}{due}."
                f"\n\nKind regards,\n{vendor}, accounts department")
    elif layout == "receipt_ocr":
        short = f"{issued:%d/%m/%y}"
        rendered["issue_date"] = short
        if rng.random() < 0.5:
            truth["document_number"] = f"R{rng.randint(100000, 999999)}"
        sym = SYMBOL[currency]
        rows = [f"{li['description']:<22}{li['amount_cents'] / 100:>8.2f}".replace(".", ",") for li in lines]
        if "absent_total" in tags:
            truth["stated_total_cents"] = None
            total_line = "TOTAL   ....  [paper torn]"
        else:
            total_line = f"TOTAL   {sym} {printed_total / 100:.2f}".replace(".", ",")
        head = [vendor.upper(), "** RECEIPT **"] + ([f"Rcpt #{truth['document_number']}"] if truth["document_number"] else [])
        text = "\n".join(head + [f"{short} 14:{rng.randint(10, 59)}"] + rows + [total_line, "CARD ****" + str(rng.randint(1000, 9999))])
    elif layout == "credit_note":
        truth["document_number"] = f"CN-{number}"
        rows = [f"| {li['description']} | {money(li['amount_cents'])} |" for li in lines]
        text = "\n".join([f"CREDIT NOTE {truth['document_number']}", vendor, f"Date: {truth['issue_date']}",
                          f"Against invoice: INV-{rng.randint(1000, 9999)}-{rng.randint(10, 99)}",
                          "Reason: goods returned damaged", "", "| Item | Credit |", "|---|---|"] + rows +
                         ["", f"TOTAL CREDIT: {money(printed_total)} {currency}"])
    elif layout == "purchase_order":
        truth["document_number"] = f"PO-{rng.randint(10000, 99999)}"
        truth["payment_terms"] = "net_30"
        deliver = issued + timedelta(days=rng.randint(5, 20))
        rendered["deliver_by"] = deliver.isoformat()          # NOT a payment due date: the label keeps due_date null
        rows = [f"| {li['description']} | {money(li['amount_cents'])} |" for li in lines]
        text = "\n".join([f"PURCHASE ORDER {truth['document_number']}", f"Buyer: {BUYER}", f"Supplier: {vendor}",
                          f"Order date: {truth['issue_date']}", f"Deliver by: {deliver.isoformat()}", "Terms: Net 30",
                          f"Currency: {currency}", "", "| Item | Amount |", "|---|---|"] + rows +
                         ["", f"ORDER TOTAL: {money(printed_total)} {currency}"])
    else:  # remittance advice: not one of the business's main types -> "other" + detail
        truth["doc_type_detail"] = "remittance advice"
        paid = [f"INV-{rng.randint(1000, 9999)}-{rng.randint(10, 99)}" for _ in lines]
        truth["line_items"] = [{"description": f"Payment of {p}", "amount_cents": li["amount_cents"]} for p, li in zip(paid, lines)]
        rows = [f"  {p}   {currency} {money(li['amount_cents'])}" for p, li in zip(paid, lines)]
        text = "\n".join(["REMITTANCE ADVICE", f"From: {BUYER}", f"To: {vendor}", f"Payment date: {truth['issue_date']}",
                          "We have paid the following invoices:"] + rows + [f"Amount paid: {currency} {money(printed_total)}"])
    if "oversized" in tags:                                    # a huge annex: the document no longer fits the context window
        log = [f"{issued + timedelta(days=i % 28)} 0{i % 9 + 1}:{i % 60:02d}  Site {'ABCD'[i % 4]}  delivery confirmed  ref {4000 + i}"
               for i in range(700)]
        text += f"\n\nANNEX A - Delivery log, {vendor}\n" + "\n".join(log)
    return Doc(doc_id, layout, text, truth, rendered, set(tags))


def make_corpus(plan: list[tuple[str, int]], seed: int, prefix: str, special: bool = True) -> list[Doc]:
    """Deterministic corpus. `special` adds the scenario tags used by the production batch."""
    rng = random.Random(seed)
    layouts = [layout for layout, n in plan for _ in range(n)]
    rng.shuffle(layouts)
    tag_plan: dict[int, set] = defaultdict(set)
    if special:
        table_idx = [i for i, lay in enumerate(layouts) if lay == "invoice_table"]
        narrative_idx = [i for i, lay in enumerate(layouts) if lay == "invoice_narrative"]
        receipt_idx = [i for i, lay in enumerate(layouts) if lay == "receipt_ocr"]
        for name, idx in [("oversized", table_idx[0:4]), ("long_items", table_idx[4:6]), ("source_conflict", table_idx[6:9]),
                          ("no_number", table_idx[9:12]), ("unclear_terms", table_idx[12:14]),
                          ("transient", table_idx[14:16]), ("expire", table_idx[16:18]),
                          ("source_conflict", narrative_idx[0:1]), ("absent_total", receipt_idx[0:3])]:
            for i in idx:
                tag_plan[i].add(name)
    else:   # calibration / sample sets: realistic noise, but no infrastructure failures
        for i, lay in enumerate(layouts):
            r = rng.random()
            if lay == "receipt_ocr" and r < 0.12:
                tag_plan[i].add("absent_total")
            if lay == "invoice_table" and r < 0.08:
                tag_plan[i].add("no_number")
    return [make_doc(f"{prefix}-{i + 1:04d}", lay, rng, tag_plan[i]) for i, lay in enumerate(layouts)]


CORPUS = make_corpus(PRODUCTION_PLAN, seed=2026, prefix="doc")
REGISTRY: dict[str, Doc] = {d.doc_id: d for d in CORPUS}      # the simulated model looks documents up here

print(len(CORPUS), "documents:", dict(Counter(d.layout for d in CORPUS)))
print("scenario tags:", dict(Counter(t for d in CORPUS for t in d.tags)))
absent = Counter(f for d in CORPUS for f in SCORED_FIELDS if d.truth[f] is None)
print("labels that are legitimately null:", dict(absent))

In [ ]:
for d in CORPUS:
    if d.layout == "invoice_narrative":
        print(f"--- {d.doc_id} ({d.layout})\n{d.text}\n")
        break
for d in CORPUS:
    if d.layout == "receipt_ocr" and "absent_total" in d.tags:
        print(f"--- {d.doc_id} ({d.layout}, tags={sorted(d.tags)})\n{d.text}\n")
        print("label:", json.dumps({k: d.truth[k] for k in SCORED_FIELDS}))
        break

> **Observe:** the receipt's label has `stated_total_cents: null`. The total is genuinely not on the paper, so `null` is the *correct* answer, not a failure. Hold on to that distinction: it decides later whether a retry can help.

> **Try it:** print one `purchase_order`. It has a "Deliver by" date. Should `due_date` be filled from it? (The label says no: a delivery date is not a payment due date. Watch how often the simulated model gets this wrong with and without few-shot examples in 3.3.)

## 3.2 Extraction schema design

One canonical contract (a Pydantic model) and two wire schemas generated for the two vendors' strict modes. Absent-able values are **nullable**; the open-ended categories use an **enum with `"other"` plus a detail string** (and `"unclear"` where ambiguity is real); `po_reference` is **optional** on Claude (left out of `required`) but must be **required + nullable** on OpenAI, whose strict mode requires every property. Confidence ranges are checked in code, because Claude's strict mode rejects `minimum`/`maximum`. [Theory](README.md#32-extraction-schema-design)

In [ ]:
DocType = Literal["invoice", "receipt", "credit_note", "purchase_order", "other"]
Currency = Literal["EUR", "USD", "GBP", "other"]
Terms = Literal["net_15", "net_30", "net_60", "due_on_receipt", "unclear", "other"]


class LineItem(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    description: str
    amount_cents: int


class FieldConfidence(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    doc_type: float
    vendor_name: float
    document_number: float
    issue_date: float
    due_date: float
    currency: float
    payment_terms: float
    stated_total_cents: float


class DocumentExtraction(BaseModel):
    """The canonical contract. Both vendors' schemas are generated to match it."""
    model_config = ConfigDict(extra="forbid", strict=True)
    doc_type: DocType
    doc_type_detail: Optional[str]
    vendor_name: str
    document_number: Optional[str]
    issue_date: Optional[str]
    due_date: Optional[str]
    currency: Currency
    currency_detail: Optional[str]
    payment_terms: Optional[Terms]
    payment_terms_detail: Optional[str]
    line_items: list[LineItem]
    stated_total_cents: Optional[int]
    po_reference: Optional[str] = None        # OPTIONAL key (Claude); OpenAI strict sends it as required + nullable
    conflict_detected: bool
    field_confidence: FieldConfidence


def _nullable(t: str, nullable: bool, description: str) -> dict:
    return {"type": [t, "null"] if nullable else t, "description": description}


def extraction_schema(provider: Literal["claude", "openai"], *, nullable: bool = True, with_other: bool = True) -> dict:
    """Strict JSON schema for one provider. nullable/with_other=False build the NAIVE schema used as a counter-example."""
    def enum_field(values: list[str], allow_null: bool, description: str) -> dict:
        vals = [v for v in values if with_other or v not in ("other", "unclear")]
        base = {"type": "string", "enum": vals}
        if allow_null and nullable:
            return {"anyOf": [base, {"type": "null"}], "description": description}
        return {**base, "description": description}

    props = {
        "doc_type": enum_field(["invoice", "receipt", "credit_note", "purchase_order", "other"], False,
                               "Document category. Use 'other' and fill doc_type_detail for anything else."),
        "doc_type_detail": _nullable("string", True, "What the document is when doc_type is 'other'; otherwise null."),
        "vendor_name": {"type": "string", "description": "Supplier/payee name exactly as printed."},
        "document_number": _nullable("string", nullable, "Invoice/receipt/credit-note/PO number as printed; null if none is printed."),
        "issue_date": _nullable("string", nullable, "Issue/payment date as ISO 8601 YYYY-MM-DD; null if not printed."),
        "due_date": _nullable("string", nullable, "PAYMENT due date as YYYY-MM-DD. A delivery date is not a due date. Null if not stated."),
        "currency": enum_field(["EUR", "USD", "GBP", "other"], False, "ISO 4217 code; 'other' + currency_detail for any other currency."),
        "currency_detail": _nullable("string", True, "ISO 4217 code when currency is 'other'; otherwise null."),
        "payment_terms": enum_field(["net_15", "net_30", "net_60", "due_on_receipt", "unclear", "other"], True,
                                    "Payment terms. 'unclear' if stated vaguely, 'other' + detail if specific but not listed, null if absent."),
        "payment_terms_detail": _nullable("string", True, "Terms as printed when payment_terms is 'other'; otherwise null."),
        "line_items": {"type": "array", "items": {
            "type": "object",
            "properties": {"description": {"type": "string"}, "amount_cents": {"type": "integer", "description": "Integer cents."}},
            "required": ["description", "amount_cents"], "additionalProperties": False}},
        "stated_total_cents": _nullable("integer", nullable, "The total exactly as PRINTED, in integer cents; null if not printed."),
        "po_reference": {"type": ["string", "null"] if provider == "openai" else "string",
                         "description": "Buyer's purchase-order reference if printed."},
        "conflict_detected": {"type": "boolean", "description": "True if the printed total differs from the sum of the line items."},
        "field_confidence": {
            "type": "object",
            "description": "Per-field confidence from 0 to 1: how directly the document states the value.",
            "properties": {f: {"type": "number"} for f in SCORED_FIELDS},
            "required": list(SCORED_FIELDS), "additionalProperties": False},
    }
    required = [k for k in props if not (provider == "claude" and k == "po_reference")]   # Claude: a truly optional key
    return {"type": "object", "properties": props, "required": required, "additionalProperties": False}


def _walk_objects(schema: dict):
    if isinstance(schema, dict):
        t = schema.get("type")
        if t == "object" or (isinstance(t, list) and "object" in t):
            yield schema
        for v in schema.values():
            if isinstance(v, dict):
                yield from _walk_objects(v)
            elif isinstance(v, list):
                for x in v:
                    yield from _walk_objects(x)


def lint_schema(schema: dict, provider: str) -> list[str]:
    """Strict-mode rules from Chapter 2 (Claude) and OpenAI Chapter 02, checked before anything is sent."""
    problems = []
    for obj in _walk_objects(schema):
        if obj.get("additionalProperties") is not False:
            problems.append(f"object with keys {sorted(obj.get('properties', {}))[:3]}... needs additionalProperties: false")
        missing = set(obj.get("properties", {})) - set(obj.get("required", []))
        if provider == "openai" and missing:
            problems.append(f"OpenAI strict: every property must be required (make it nullable instead): {sorted(missing)}")
    text = json.dumps(schema)
    if provider == "claude":
        for kw in ("minimum", "maximum", "minLength", "maxLength", "multipleOf"):
            if f'"{kw}"' in text:
                problems.append(f"Claude strict: '{kw}' is not supported; validate it in code")
        # The limits are totals over every property of every strict schema in the request, nested objects included.
        params = [(obj, name, p) for obj in _walk_objects(schema) for name, p in obj.get("properties", {}).items()]
        unions = sum(1 for _, _, p in params if "anyOf" in p or isinstance(p.get("type"), list))
        optional = sum(1 for obj, name, _ in params if name not in obj.get("required", []))
        if unions > 16:
            problems.append(f"Claude strict: {unions} union-typed parameters > 16")
        if optional > 24:
            problems.append(f"Claude strict: {optional} optional parameters > 24")
    return problems


CLAUDE_SCHEMA, OPENAI_SCHEMA = extraction_schema("claude"), extraction_schema("openai")
for name, s in [("claude", CLAUDE_SCHEMA), ("openai", OPENAI_SCHEMA)]:
    print(f"{name:6} required={len(s['required'])}/{len(s['properties'])} lint={lint_schema(s, name) or 'OK'}")
print("Claude schema used on OpenAI:", lint_schema(CLAUDE_SCHEMA, "openai"))
print("with a 'minimum' on Claude:", lint_schema({**CLAUDE_SCHEMA, "properties": {**CLAUDE_SCHEMA["properties"],
      "x": {"type": "number", "minimum": 0}}}, "claude"))
_nested = {"type": "object", "additionalProperties": False, "required": [],
           "properties": {f"extra_{i}": {"type": ["string", "null"]} for i in range(25)}}
print("25 optional nullable fields NESTED in one object:", lint_schema({**CLAUDE_SCHEMA, "properties": {
      **CLAUDE_SCHEMA["properties"], "extras": _nested}}, "claude"))

> **Observe:** the same contract needs two wire forms. The Claude schema keeps `po_reference` out of `required` (a truly optional key); OpenAI's strict mode rejects that and wants **required + nullable**. Either way the *meaning* is identical: "not printed" is expressed as absence or `null`, never as an invented value. Claude's complexity limits (24 optional parameters, 16 union-typed parameters) count **every** property in **every** strict schema of the request, nested objects included, which is why the last line is rejected even though the root object looks small.

### The simulated model, and what a naive schema does to it (offline)

`simulated_extraction()` reads the request the way a model would see it (system prompt, document, schema, any feedback) and returns an extraction. Its error profiles are the patterns the exam guide and the chapters describe: format slips on dates, totals missed in narrative text, decimal-comma misreads on receipts, overconfident OCR mistakes, delivery dates mistaken for due dates, and **fabrication when the schema has no way to say "absent"**.

In [ ]:
FAKE_CONTEXT_TOKENS = 4_000      # fake: a tiny context window so "prompt is too long" is cheap to trigger
DOC_TAG = re.compile(r'<document id="(?P<id>[\w-]+)"(?: chunk="(?P<chunk>\d+)")?')
FEEDBACK_LINE = re.compile(r"^- (?P<field>[a-z_]+): (?P<code>[a-z_]+)", re.M)


def approx_tokens(text: str) -> int:
    """~4 characters per token. Planning only; use count_tokens / input_tokens.count for real numbers."""
    return max(1, math.ceil(len(text) / 4))


def _rng(*parts) -> random.Random:
    return random.Random(int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:12], 16))


# (name, field, layouts, P(error) without few-shot, P(error) with few-shot, fixable by feedback?)
ERROR_PROFILES = [
    ("date_not_iso", "issue_date", {"invoice_narrative", "receipt_ocr"}, 0.45, 0.05, True),
    ("total_missed_in_prose", "stated_total_cents", {"invoice_narrative"}, 0.55, 0.06, True),
    ("decimal_comma_misread", "line_items", {"receipt_ocr"}, 0.30, 0.05, True),
    ("other_without_detail", "detail", {"remittance", "invoice_table", "invoice_narrative"}, 0.25, 0.08, True),
    ("conflict_not_flagged", "conflict_detected", set(LAYOUT_DOC_TYPE), 0.30, 0.10, True),
    # "valid but wrong": schema-valid, passes validation, only calibration / review / sampling can catch these
    ("ocr_vendor_typo", "vendor_name", {"receipt_ocr"}, 0.16, 0.12, False),
    ("day_month_swap", "issue_date", {"receipt_ocr"}, 0.14, 0.10, False),
    ("number_transposed", "document_number", {"invoice_narrative"}, 0.10, 0.06, False),
    ("deliver_by_as_due", "due_date", {"purchase_order"}, 0.35, 0.05, False),
    ("po_as_invoice", "doc_type", {"purchase_order"}, 0.08, 0.03, False),
    ("terms_misread", "payment_terms", {"invoice_narrative"}, 0.12, 0.06, False),
    ("due_date_slip", "due_date", {"invoice_table"}, 0.03, 0.02, False),
]
HIGH_CONF = {"invoice_table": 0.97, "credit_note": 0.95, "purchase_order": 0.94, "remittance": 0.92,
             "invoice_narrative": 0.91, "receipt_ocr": 0.93}


def _allows_null(schema: dict, key: str) -> bool:
    p = schema.get("properties", {}).get(key, {})
    t = p.get("type")
    return (isinstance(t, list) and "null" in t) or any(o.get("type") == "null" for o in p.get("anyOf", []))


def _enum_of(schema: dict, key: str) -> list:
    p = schema.get("properties", {}).get(key, {})
    return p.get("enum") or next((o.get("enum") for o in p.get("anyOf", []) if o.get("enum")), [])


def simulated_extraction(document: str, system: str, schema: dict, feedback: str = "") -> dict:
    """fake: plays the model. Starts from the document's label and applies the error profiles above."""
    m = DOC_TAG.search(document)
    doc = REGISTRY[m["id"]]
    chunk = int(m["chunk"] or 0)
    t, lay = copy.deepcopy(doc.truth), doc.layout
    few_shot = "<examples>" in system
    flagged = {(mm["field"], mm["code"]) for mm in FEEDBACK_LINE.finditer(feedback)}
    flagged_fields = {f for f, _ in flagged}
    pressure = "must not be null" in feedback          # the anti-pattern: demanding a value the source lacks
    out = copy.deepcopy(t)
    wrong: set[str] = set()

    if chunk > 0:                                      # an annex-only chunk: header facts are not visible here
        out.update({k: None for k in ("document_number", "issue_date", "due_date", "payment_terms",
                                      "payment_terms_detail", "stated_total_cents", "po_reference")})
        out.update({"line_items": [], "conflict_detected": False})
    for name, fld, layouts, p_plain, p_few, fixable in ERROR_PROFILES:
        if lay not in layouts or chunk > 0:
            continue
        if _rng(doc.doc_id, name, few_shot).random() >= (p_few if few_shot else p_plain):
            continue
        targets = {"detail": {"doc_type_detail", "payment_terms_detail", "currency_detail"},
                   "line_items": {"line_items", "stated_total_cents"}}.get(fld, {fld})
        if fixable and targets & flagged_fields:
            continue                                   # feedback pointed at it and the source supports a fix
        if name == "date_not_iso":
            out["issue_date"] = doc.rendered["issue_date"]
        elif name == "total_missed_in_prose":
            out["stated_total_cents"] = None
        elif name == "decimal_comma_misread":
            out["line_items"][0] = {**out["line_items"][0], "amount_cents": out["line_items"][0]["amount_cents"] * 100}
        elif name == "other_without_detail":
            for key in ("doc_type", "payment_terms", "currency"):
                if out.get(key) == "other":
                    out[f"{key}_detail"] = None
        elif name == "conflict_not_flagged":
            out["conflict_detected"] = False
        elif name == "ocr_vendor_typo":
            out["vendor_name"] = out["vendor_name"].replace("o", "0", 1).replace("l", "1", 1)
        elif name == "day_month_swap":
            d = date.fromisoformat(t["issue_date"])
            if d.day <= 12 and d.day != d.month:
                out["issue_date"] = date(d.year, d.day, d.month).isoformat()
        elif name == "number_transposed" and t["document_number"]:
            n = t["document_number"]
            out["document_number"] = n[:-2] + n[-1] + n[-2]
        elif name == "deliver_by_as_due":
            out["due_date"] = doc.rendered["deliver_by"]
        elif name == "po_as_invoice":
            out["doc_type"] = "invoice"
        elif name == "terms_misread" and t["payment_terms"] not in (None, "unclear"):
            out["payment_terms"], out["payment_terms_detail"] = "unclear", None
        elif name == "due_date_slip" and t["due_date"]:
            out["due_date"] = (date.fromisoformat(t["due_date"]) + timedelta(days=1)).isoformat()

    # Fabrication: a field the source lacks, but the schema gives no way to say so (or feedback demands a value)
    for key in ("document_number", "issue_date", "due_date", "stated_total_cents"):
        if out[key] is None and (not _allows_null(schema, key) or (pressure and key in flagged_fields)):
            out[key] = {"document_number": f"INV-{_rng(doc.doc_id, key).randint(1000, 9999)}-01",
                        "issue_date": t["issue_date"] or "2026-01-01",
                        "due_date": (date.fromisoformat(t["issue_date"]) + timedelta(days=30)).isoformat(),
                        "stated_total_cents": sum(li["amount_cents"] for li in out["line_items"])}[key]
    if out["payment_terms"] is None and not _allows_null(schema, "payment_terms"):
        out["payment_terms"] = "net_30"
    for key in ("doc_type", "currency", "payment_terms"):     # no "other"/"unclear" bucket: forced into a wrong one
        allowed = _enum_of(schema, key)
        if out[key] is not None and allowed and out[key] not in allowed:
            out[key] = {"doc_type": "invoice", "currency": "EUR", "payment_terms": "net_30"}[key]
            out[f"{key}_detail"] = None
    if "po_reference" not in schema.get("required", []) and out["po_reference"] is None:
        out.pop("po_reference")                        # Claude: an optional key is simply left out

    conf = {}
    for f in SCORED_FIELDS:
        r = _rng(doc.doc_id, f, few_shot, feedback[:40])
        correct = out.get(f) == t[f]
        if correct:
            c = HIGH_CONF[lay] - abs(r.gauss(0, 0.035))
        elif lay == "receipt_ocr":
            c = r.gauss(0.88, 0.05)                    # OCR mistakes look confident: the reason to calibrate
        else:
            c = r.gauss(0.74, 0.10)
        conf[f] = round(min(0.99, max(0.05, c)), 2)
    out["field_confidence"] = conf
    return out


def doc_block(doc_id: str, text: str, chunk: int | None = None) -> str:
    attr = "" if chunk is None else f' chunk="{chunk}"'
    return f'<document id="{doc_id}"{attr}>\n{text}\n</document>'


# Naive vs good schema on the documents that have absent fields or out-of-list categories
NAIVE = extraction_schema("claude", nullable=False, with_other=False)
probe = [d for d in CORPUS if d.truth["document_number"] is None or d.truth["doc_type"] == "other"
         or d.truth["currency"] == "other"][:12]
fabricated = Counter()
for d in probe:
    naive = simulated_extraction(doc_block(d.doc_id, d.text), "", NAIVE)
    good = simulated_extraction(doc_block(d.doc_id, d.text), "", CLAUDE_SCHEMA)
    for f in ("document_number", "due_date", "stated_total_cents", "doc_type", "currency"):
        if naive[f] != d.truth[f]:
            fabricated["naive:" + f] += 1
        if good[f] != d.truth[f]:
            fabricated["good:" + f] += 1
print(f"{len(probe)} documents with absent values or unlisted categories")
for f in ("document_number", "due_date", "stated_total_cents", "doc_type", "currency"):
    print(f"  {f:20} wrong with NAIVE schema: {fabricated['naive:' + f]:2}   with nullable + 'other' schema: {fabricated['good:' + f]:2}")

> **Observe:** with the naive schema (every field a non-null string, closed enums) the "model" fills every gap with a plausible invention (`INV-xxxx-01`, a due date 30 days out, a remittance advice filed as an invoice, CHF reported as EUR). Nothing in validation can catch these: they are schema-valid. The nullable + `"other"` schema removes that whole class of error.

> **Try it:** add `"unclear"` to `doc_type`'s enum and describe when to use it. Which documents in this corpus would honestly deserve it?

## 3.3 Prompt and few-shot examples for varied formats

The same facts appear as table rows, inside prose, and as OCR text with decimal commas. Detailed instructions alone leave the model inconsistent on the unusual layouts; **2–4 targeted few-shot examples** that show the tricky cases (a total stated in a sentence, `12,50` meaning 12.50, a delivery date that is not a due date) fix most of it. Measure that on a **labeled sample before** the big batch run, never after. [Theory](README.md#33-prompt-and-few-shot-examples-for-varied-formats)

In [ ]:
BASE_INSTRUCTIONS = """You extract accounts-payable data for Harbor & Pine Facilities.
Rules:
- Return exactly one extraction that matches the schema. Use null for any value the document does not state; never guess.
- Dates: ISO 8601 (YYYY-MM-DD). Amounts: integer cents. Currency: ISO 4217; 'other' + currency_detail for unlisted codes.
- stated_total_cents is the total exactly as printed. Do not compute it. If the line items do not add up to it, keep the printed value and set conflict_detected to true.
- due_date is a PAYMENT due date only. Delivery or service dates are not due dates.
- Use 'other' plus the matching *_detail field for categories outside the list; use 'unclear' for vague payment terms.
- field_confidence: 0-1 per field, how directly the document states the value (lower it for inferred, faint or ambiguous values)."""

FEW_SHOT = """<examples>
<example layout="narrative letter">
Text: "... the sum of EUR 1,250.00 is payable within 30 days, that is by 4 May 2026."
Extract: stated_total_cents 125000, payment_terms "net_30", due_date "2026-05-04". The total sits inside a sentence; it is still the printed total.
</example>
<example layout="OCR receipt">
Text: "batteries aa x24      12,50 / TOTAL   € 31,40 / 07/03/26"
Extract: amount_cents 1250 ("12,50" uses a decimal comma: 12.50, not 1,250.00), stated_total_cents 3140, issue_date "2026-03-07" (day/month/year), currency "EUR".
If the total line is torn or illegible: stated_total_cents null. Do not add up the lines instead.
</example>
<example layout="purchase order">
Text: "Order date: 2026-02-10 / Deliver by: 2026-02-24 / Terms: Net 30"
Extract: doc_type "purchase_order", due_date null (a delivery date is not a payment due date), payment_terms "net_30".
</example>
<example layout="remittance advice">
Text: "REMITTANCE ADVICE ... Amount paid: CHF 980.00"
Extract: doc_type "other", doc_type_detail "remittance advice", currency "other", currency_detail "CHF".
</example>
</examples>"""

SYSTEM_PLAIN = BASE_INSTRUCTIONS
SYSTEM_FEW_SHOT = BASE_INSTRUCTIONS + "\n\n" + FEW_SHOT


def field_correct(pred: dict, truth: dict, f: str) -> bool:
    a, b = pred.get(f), truth.get(f)
    if isinstance(a, str) and isinstance(b, str):
        return a.strip().casefold() == b.strip().casefold()
    return a == b


# A labeled SAMPLE (separate seed, no infrastructure failures) used to choose the prompt before the big run
SAMPLE = make_corpus([(lay, 10) for lay, _ in PRODUCTION_PLAN], seed=7, prefix="smp", special=False)
REGISTRY.update({d.doc_id: d for d in SAMPLE})


def validation_issues_quick(out: dict) -> list[str]:
    """A minimal check for this section; 3.4 builds the full validator."""
    iso = re.compile(r"^\d{4}-\d{2}-\d{2}$")
    issues = [f for f in ("issue_date", "due_date") if out.get(f) and not iso.match(out[f])]
    if out.get("stated_total_cents") is None and out["doc_type"] in ("invoice", "credit_note"):
        issues.append("missing_total")
    if out.get("stated_total_cents") is not None and not out["conflict_detected"] and \
            sum(li["amount_cents"] for li in out["line_items"]) != out["stated_total_cents"]:
        issues.append("sum_mismatch")
    return issues


def evaluate_prompt(system: str, docs: list[Doc]) -> dict[str, dict[str, float]]:
    by_layout: dict[str, list] = defaultdict(list)
    for d in docs:
        out = simulated_extraction(doc_block(d.doc_id, d.text), system, CLAUDE_SCHEMA)
        valid = not validation_issues_quick(out)
        acc = sum(field_correct(out, d.truth, f) for f in SCORED_FIELDS) / len(SCORED_FIELDS)
        by_layout[d.layout].append((valid, acc))
    return {lay: {"first_pass_valid": sum(v for v, _ in rows) / len(rows), "field_accuracy": sum(a for _, a in rows) / len(rows)}
            for lay, rows in by_layout.items()}


plain, few = evaluate_prompt(SYSTEM_PLAIN, SAMPLE), evaluate_prompt(SYSTEM_FEW_SHOT, SAMPLE)
print(f"{'layout':18} {'valid: plain':>13} {'few-shot':>9}   {'accuracy: plain':>16} {'few-shot':>9}")
for lay, _ in PRODUCTION_PLAN:
    print(f"{lay:18} {plain[lay]['first_pass_valid']:13.0%} {few[lay]['first_pass_valid']:9.0%}   "
          f"{plain[lay]['field_accuracy']:16.1%} {few[lay]['field_accuracy']:9.1%}")

> **Observe:** the few-shot prompt barely moves the clean tabular invoices (they were already fine) but lifts first-pass validity on the narrative letters and receipts, and halves the purchase-order due-date mistakes in this small sample (2 to 1). That gap is the whole argument for refining on a stratified sample: on 100,000 documents, every point of first-pass validity is 1,000 fewer resubmissions. Notice also what the examples do **not** fix: receipt OCR typos stay. Those need review routing (3.7), not a better prompt.

> **Try it:** remove the receipt example from `FEW_SHOT` and re-run. Which numbers move?

## 3.4 Validation and retry with error feedback

Strict schemas remove **syntax** errors. They do not remove **semantic** ones: a date in the wrong format inside a valid string, lines that do not add up, `"other"` with no detail, a confidence of 95 instead of 0.95. The validator below returns specific, field-level issues. A retry sends back the **document, the failed extraction and those specific errors**. It fixes format and structural mistakes; it cannot conjure a value that is not in the source, and pressuring the model to try produces fabrication. [Theory](README.md#34-validation-and-retry-with-error-feedback)

In [ ]:
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
TOTAL_REQUIRED_FOR = {"invoice", "receipt", "credit_note"}       # the business needs a total for these


@dataclass
class Issue:
    field: str
    code: str
    message: str
    retryable: bool        # True: the model can fix it from what it has. False: needs a human or a different input


def validate_extraction(data: dict) -> list[Issue]:
    try:
        ext = DocumentExtraction.model_validate(data)
    except ValidationError as e:
        return [Issue(".".join(map(str, err["loc"])) or "root", "schema", err["msg"], True) for err in e.errors()[:5]]
    issues = []
    for f in ("issue_date", "due_date"):
        v = getattr(ext, f)
        if v is not None:
            if not ISO_DATE.match(v):
                issues.append(Issue(f, "date_format", f"'{v}' is not ISO 8601 YYYY-MM-DD", True))
            else:
                try:
                    date.fromisoformat(v)
                except ValueError:
                    issues.append(Issue(f, "date_format", f"'{v}' is not a real calendar date", True))
    for key in ("doc_type", "currency", "payment_terms"):
        if getattr(ext, key) == "other" and not getattr(ext, f"{key}_detail"):
            issues.append(Issue(f"{key}_detail", "other_without_detail", f"{key} is 'other' but {key}_detail is empty", True))
    bad_conf = [f for f, c in ext.field_confidence.model_dump().items() if not 0 <= c <= 1]
    if bad_conf:
        issues.append(Issue("field_confidence", "confidence_range", f"values must be within 0-1: {bad_conf}", True))
    lines_sum = sum(li.amount_cents for li in ext.line_items)
    if ext.stated_total_cents is None:
        if ext.doc_type in TOTAL_REQUIRED_FOR:
            issues.append(Issue("stated_total_cents", "missing_total",
                                "no printed total was extracted; re-read the document (keep null if it truly has none)", True))
    elif lines_sum != ext.stated_total_cents and not ext.conflict_detected:
        issues.append(Issue("line_items", "sum_mismatch",
                            f"line items sum to {lines_sum} but stated_total_cents is {ext.stated_total_cents}; re-check each "
                            "amount, and if the document itself is inconsistent set conflict_detected to true", True))
    return issues


def feedback_text(issues: list[Issue], pressure: bool = False) -> str:
    """The specific errors, one per line, in a stable format a person (or a log query) can read."""
    lines = [f"- {i.field}: {i.code}: {i.message}" for i in issues]
    tail = ("Every field is required and must not be null." if pressure else
            "Correct only the affected fields. If a value is truly not in the document, keep it null.")
    return "Your extraction failed validation:\n" + "\n".join(lines) + "\n" + tail


examples = {d.layout: d for d in SAMPLE}
for lay in ("invoice_narrative", "receipt_ocr", "remittance"):
    d = examples[lay]
    out = simulated_extraction(doc_block(d.doc_id, d.text), SYSTEM_PLAIN, CLAUDE_SCHEMA)
    print(f"{d.doc_id} {lay:18}", [(i.field, i.code) for i in validate_extraction(out)] or "valid")
print("schema error example:", [(i.field, i.code) for i in validate_extraction({"doc_type": "memo"})][:3])

### The retry loop through the real SDKs (offline)

Before the batch section, here is the follow-up request in both vendors' wire formats, sent to the fakes through the **real** clients. Claude gets a pre-built conversation (document → failed extraction → specific errors); OpenAI gets the same three turns as `input` items. Both use the vendor's structured-output mechanism: `output_config.format` on Claude and `text.format` with `strict: true` on OpenAI. Run the toolkit cell first.

### Offline toolkit: fake batch services behind the real SDKs (run once)

In [ ]:
_ids = count(1)
CUSTOM_ID_RE = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")
REJECTS_FORCED_TOOL = {"claude-sonnet-5-5", "claude-opus-5-5", "claude-fable-5-1"}   # per Chapter 2 (current docs)
TRANSIENT_ONCE = {d.doc_id for d in CORPUS if "transient" in d.tags}
EXPIRE_ONCE = {d.doc_id for d in CORPUS if "expire" in d.tags}


def base_id(custom_id: str) -> str:
    return re.sub(r"-c\d+$", "", custom_id)


def _iso(t: datetime | None) -> str | None:
    return None if t is None else t.isoformat().replace("+00:00", "Z")


def claude_view_of(params: dict) -> tuple[str, str, str]:
    """(system, document, feedback) as the model would read them from a Messages request."""
    system = params.get("system") or ""
    if isinstance(system, list):
        system = "\n".join(b.get("text", "") for b in system)
    texts = []
    for msg in params.get("messages", []):
        content = msg["content"]
        if isinstance(content, str):
            texts.append((msg["role"], content))
        else:
            for b in content:
                if b.get("type") == "text":
                    texts.append((msg["role"], b["text"]))
                elif b.get("type") == "tool_result":
                    inner = b.get("content")
                    texts.append(("user", inner if isinstance(inner, str) else " ".join(x.get("text", "") for x in inner)))
    document = next((t for r, t in texts if r == "user" and "<document" in t), "")
    feedback = "\n".join(t for r, t in texts[1:] if r == "user")
    return system, document, feedback


def claude_message(content: list, stop_reason: str, model: str, input_tokens: int, output_tokens: int) -> dict:
    return {"id": f"msg_fake{next(_ids):05d}", "type": "message", "role": "assistant", "model": model,
            "content": content, "stop_reason": stop_reason, "stop_sequence": None, "stop_details": None,
            "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens,
                      "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0}}


def claude_responder(custom_id: str, params: dict, attempt: int) -> dict:
    """fake: one Messages request -> {"message"} | {"error": (type, msg)} | {"expired": True}."""
    doc_id = base_id(custom_id)
    if doc_id in TRANSIENT_ONCE and attempt == 0:
        return {"error": ("overloaded_error", "Overloaded")}
    if doc_id in EXPIRE_ONCE and attempt == 0:
        return {"expired": True}
    system, document, feedback = claude_view_of(params)
    prompt_tokens = approx_tokens(system + document + feedback)
    if prompt_tokens > FAKE_CONTEXT_TOKENS:
        return {"error": ("invalid_request_error", f"prompt is too long: {prompt_tokens} tokens > {FAKE_CONTEXT_TOKENS} maximum")}
    choice = params.get("tool_choice") or {}
    if choice.get("type") in ("any", "tool") and params["model"] in REJECTS_FORCED_TOOL:
        return {"error": ("invalid_request_error", f"tool_choice {choice['type']} is not supported on {params['model']}")}
    tools = params.get("tools") or []
    schema = tools[0]["input_schema"] if tools else params["output_config"]["format"]["schema"]
    data = simulated_extraction(document, system, schema, feedback)
    payload = json.dumps(data)
    out_tokens = approx_tokens(payload)
    if out_tokens > params["max_tokens"]:                       # cut off mid-JSON
        return {"message": claude_message([{"type": "text", "text": payload[: params["max_tokens"] * 4]}] if not tools else [],
                                          "max_tokens", params["model"], prompt_tokens, params["max_tokens"])}
    if tools:
        block = {"type": "tool_use", "id": f"toolu_fake{next(_ids):05d}", "name": tools[0]["name"], "input": data}
        return {"message": claude_message([block], "tool_use", params["model"], prompt_tokens, out_tokens)}
    return {"message": claude_message([{"type": "text", "text": payload}], "end_turn", params["model"], prompt_tokens, out_tokens)}


@dataclass
class _ClaudeBatch:
    id: str
    requests: list[dict]
    created_at: datetime
    status: str = "in_progress"
    ended_at: datetime | None = None
    results: list[dict] = field(default_factory=list)


class FakeClaudeBatches:
    """Offline Message Batches API (+ messages.create, count_tokens). `.client` is a REAL anthropic client.
    Virtual clock: `.sleep(s)` moves time; a batch ends `duration_minutes` after creation."""

    def __init__(self, responder=claude_responder, *, duration_minutes: float = 55, seed: int = 7):
        self.responder, self.duration = responder, timedelta(minutes=duration_minutes)
        self.rng, self.now = random.Random(seed), datetime(2026, 10, 3, 18, 0, tzinfo=timezone.utc)
        self.batches: dict[str, _ClaudeBatch] = {}
        self.attempts: Counter = Counter()
        self.log: list[str] = []
        self.client = anthropic.Anthropic(api_key="offline", max_retries=0,
                                          http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(self._handle)))

    def sleep(self, seconds: float) -> None:
        self.now += timedelta(seconds=seconds)

    @staticmethod
    def _error(status: int, etype: str, msg: str) -> httpx2.Response:
        return httpx2.Response(status, json={"type": "error", "error": {"type": etype, "message": msg}})

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        path, method = request.url.path, request.method
        self.log.append(f"{method} {path}")
        body = json.loads(request.content) if request.content else {}
        if path == "/v1/messages/batches" and method == "POST":
            reqs = body.get("requests", [])
            ids = [r.get("custom_id", "") for r in reqs]
            if not reqs or len(reqs) > 100_000 or len(set(ids)) != len(ids) or not all(CUSTOM_ID_RE.match(i) for i in ids):
                return self._error(400, "invalid_request_error", "1-100,000 requests with unique, legal custom_ids")
            b = _ClaudeBatch(f"msgbatch_fake{next(_ids):05d}", reqs, self.now)
            self.batches[b.id] = b
            return httpx2.Response(200, json=self._view(b))
        if path == "/v1/messages/count_tokens":
            system, document, feedback = claude_view_of(body)
            return httpx2.Response(200, json={"input_tokens": approx_tokens(system + document + feedback)})
        if path == "/v1/messages" and method == "POST":
            out = self.responder("sync", body, 1)              # sync calls never hit the one-off batch failures
            if "error" in out:
                etype, msg = out["error"]
                return self._error(400 if etype == "invalid_request_error" else 529, etype, msg)
            return httpx2.Response(200, json=out["message"])
        m = re.fullmatch(r"/v1/messages/batches/([^/]+)(/results)?", path)
        if m and m[1] in self.batches:
            b = self.batches[m[1]]
            self._advance(b)
            if m[2]:
                if b.status != "ended":
                    return self._error(400, "invalid_request_error", "Batch has not ended yet")
                return httpx2.Response(200, content=("\n".join(json.dumps(r) for r in b.results) + "\n").encode())
            return httpx2.Response(200, json=self._view(b))
        return self._error(404, "not_found_error", f"fake has no route for {method} {path}")

    def _advance(self, b: _ClaudeBatch) -> None:
        if b.status == "ended" or self.now - b.created_at < min(self.duration, timedelta(hours=24)):
            return
        for req in b.requests:
            cid = req["custom_id"]
            out = self.responder(cid, req["params"], self.attempts[cid])
            self.attempts[cid] += 1
            if out.get("expired"):
                result = {"type": "expired"}
            elif "error" in out:
                result = {"type": "errored", "error": {"type": "error", "error": {"type": out["error"][0], "message": out["error"][1]}}}
            else:
                result = {"type": "succeeded", "message": out["message"]}
            b.results.append({"custom_id": cid, "result": result})
        self.rng.shuffle(b.results)                               # results are NOT in request order
        b.status, b.ended_at = "ended", self.now

    def _view(self, b: _ClaudeBatch) -> dict:
        counts = {"processing": 0, "succeeded": 0, "errored": 0, "canceled": 0, "expired": 0}
        if b.status == "ended":
            for r in b.results:
                counts[r["result"]["type"]] += 1
        else:
            counts["processing"] = len(b.requests)                # outcome counters stay 0 until the batch ends
        return {"id": b.id, "type": "message_batch", "processing_status": b.status, "request_counts": counts,
                "created_at": _iso(b.created_at), "expires_at": _iso(b.created_at + timedelta(hours=24)),
                "ended_at": _iso(b.ended_at), "cancel_initiated_at": None, "archived_at": None,
                "results_url": f"https://api.anthropic.com/v1/messages/batches/{b.id}/results" if b.status == "ended" else None}


def openai_view_of(body: dict) -> tuple[str, str, str]:
    system = body.get("instructions") or ""
    items = body.get("input")
    if isinstance(items, str):
        return system, items, ""
    texts = []
    for it in items:
        c = it.get("content")
        texts.append((it.get("role"), c if isinstance(c, str) else " ".join(p.get("text", "") for p in c)))
    document = next((t for r, t in texts if r == "user" and "<document" in t), "")
    return system, document, "\n".join(t for r, t in texts[1:] if r == "user")


def openai_response(model: str, text: str | None, *, input_tokens: int, output_tokens: int,
                    status: str = "completed", reason: str | None = None) -> dict:
    output = [] if text is None else [{"type": "message", "id": f"msg_{next(_ids):05d}", "role": "assistant", "status": "completed",
                                       "content": [{"type": "output_text", "text": text, "annotations": []}]}]
    data = {"id": f"resp_{next(_ids):05d}", "object": "response", "created_at": 1_790_000_000, "model": model, "status": status,
            "incomplete_details": {"reason": reason} if reason else None, "output": output, "parallel_tool_calls": True,
            "tool_choice": "auto", "tools": [], "service_tier": "default",
            "usage": {"input_tokens": input_tokens, "input_tokens_details": {"cached_tokens": 0, "cache_write_tokens": 0},
                      "output_tokens": output_tokens, "output_tokens_details": {"reasoning_tokens": 32},
                      "total_tokens": input_tokens + output_tokens}}
    return Response.model_validate(data).model_dump(mode="json", exclude_none=True)   # the SDK's own type checks the shape


def openai_responder(custom_id: str, body: dict, attempt: int) -> tuple[int, dict] | str:
    """fake: one /v1/responses body -> (status_code, body), or "expired"."""
    doc_id = base_id(custom_id)
    if doc_id in TRANSIENT_ONCE and attempt == 0:
        return 500, {"error": {"message": "The server had an error while processing your request.", "type": "server_error",
                               "param": None, "code": None}}
    if doc_id in EXPIRE_ONCE and attempt == 0:
        return "expired"
    system, document, feedback = openai_view_of(body)
    prompt_tokens = approx_tokens(system + document + feedback)
    if prompt_tokens > FAKE_CONTEXT_TOKENS:          # fake: illustrative error code
        return 400, {"error": {"message": f"Input is {prompt_tokens} tokens, over the context window.",
                               "type": "invalid_request_error", "param": "input", "code": "context_length_exceeded"}}
    schema = body["text"]["format"]["schema"]
    payload = json.dumps(simulated_extraction(document, system, schema, feedback))
    out_tokens = approx_tokens(payload) + 32                  # + a few reasoning tokens (billed as output)
    if out_tokens > body.get("max_output_tokens", 10**9):
        return 200, openai_response(body["model"], None, input_tokens=prompt_tokens, output_tokens=body["max_output_tokens"],
                                    status="incomplete", reason="max_output_tokens")
    return 200, openai_response(body["model"], payload, input_tokens=prompt_tokens, output_tokens=out_tokens)


class FakeOpenAIBatches:
    """Offline Files + Batches + Responses. `.client` is a REAL openai client. Each batches.retrieve advances
    a batch one step; `.sleep(s)` only moves the virtual clock (for SLA timing)."""

    def __init__(self, responder=openai_responder, *, lines_per_poll: int = 40, seed: int = 7):
        self.responder, self.lines_per_poll, self.rng = responder, lines_per_poll, random.Random(seed)
        self.now = datetime(2026, 10, 3, 18, 0, tzinfo=timezone.utc)
        self.files: dict[str, bytes] = {}
        self.batches: dict[str, dict] = {}
        self.attempts: Counter = Counter()
        self.log: list[str] = []
        self.client = OpenAI(api_key="sk-offline", base_url="https://fake-openai.invalid/v1", max_retries=0,
                             http_client=httpx2.Client(transport=httpx2.MockTransport(self._handle)))

    def sleep(self, seconds: float) -> None:
        self.now += timedelta(seconds=seconds)

    def _id(self, prefix: str) -> str:
        return f"{prefix}_{next(_ids):05d}"

    def _store(self, records: list[dict]) -> str | None:
        if not records:
            return None
        fid = self._id("file")
        self.files[fid] = "".join(json.dumps(r) + "\n" for r in records).encode()
        return fid

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        request.read()
        path = request.url.path.removeprefix("/v1")
        self.log.append(f"{request.method} {path}")
        if request.method == "POST" and path == "/files":
            raw = b"Content-Type: " + request.headers["content-type"].encode() + b"\r\n\r\n" + request.content
            parts = {p.get_param("name", header="content-disposition"): p for p in BytesParser(policy=HTTP).parsebytes(raw).iter_parts()}
            content = parts["file"].get_payload(decode=True)
            fid = self._id("file")
            self.files[fid] = content
            return httpx2.Response(200, json={"id": fid, "object": "file", "bytes": len(content), "created_at": 1_790_000_000,
                                              "status": "processed", "filename": parts["file"].get_filename() or "batch.jsonl",
                                              "purpose": parts["purpose"].get_content().strip()})
        if m := re.fullmatch(r"/files/([\w-]+)/content", path):
            return httpx2.Response(200, content=self.files[m[1]])
        if request.method == "POST" and path == "/batches":
            params = json.loads(request.content)
            lines = [json.loads(x) for x in self.files[params["input_file_id"]].decode().splitlines() if x.strip()]
            models = {line["body"]["model"] for line in lines}
            errors = [] if len(models) == 1 else [{"code": "mixed_models", "line": None, "param": "body.model",     # fake: illustrative
                                                   "message": f"One model per input file; found {sorted(models)}."}]
            b = {"id": self._id("batch"), "object": "batch", "endpoint": params["endpoint"], "completion_window": "24h",
                 "input_file_id": params["input_file_id"], "created_at": 1_790_000_000, "status": "validating",
                 "metadata": params.get("metadata"), "request_counts": {"total": len(lines), "completed": 0, "failed": 0},
                 "_lines": lines, "_errors": errors, "_done": [], "_deferred": []}
            self.batches[b["id"]] = b
            return httpx2.Response(200, json={k: v for k, v in b.items() if not k.startswith("_")})
        if m := re.fullmatch(r"/batches/([\w-]+)", path):
            b = self._advance(self.batches[m[1]])
            return httpx2.Response(200, json={k: v for k, v in b.items() if not k.startswith("_")})
        if request.method == "POST" and path == "/responses":
            res = self.responder("sync", json.loads(request.content), 1)
            status, body = (200, res) if isinstance(res, dict) else res
            return httpx2.Response(status, json=body)
        return httpx2.Response(404, json={"error": {"message": f"fake: no route {path}", "type": "invalid_request_error"}})

    def _advance(self, b: dict) -> dict:
        if b["status"] == "validating":
            if b["_errors"]:
                b["status"], b["errors"] = "failed", {"object": "list", "data": b["_errors"]}
            else:
                b["status"] = "in_progress"
        elif b["status"] == "in_progress":
            pending = b["_lines"][len(b["_done"]) + len(b["_deferred"]):][: self.lines_per_poll]
            for line in pending:
                cid = line["custom_id"]
                res = self.responder(cid, line["body"], self.attempts[cid])
                self.attempts[cid] += 1
                if res == "expired":
                    b["_deferred"].append(line)                       # never ran inside the 24 h window
                else:
                    b["_done"].append((line, *res))
            ok = sum(code == 200 for _, code, _ in b["_done"])
            b["request_counts"] = {"total": len(b["_lines"]), "completed": ok, "failed": len(b["_done"]) - ok}
            if len(b["_done"]) + len(b["_deferred"]) == len(b["_lines"]):
                b["status"] = "finalizing"
        elif b["status"] == "finalizing":
            self._finish(b)
        return b

    def _finish(self, b: dict) -> None:
        out, err = [], []
        for line, code, body in b["_done"]:
            rec = {"id": self._id("batch_req"), "custom_id": line["custom_id"],
                   "response": {"status_code": code, "request_id": self._id("req"), "body": body}, "error": None}
            (out if code == 200 else err).append(rec)
        for line in b["_deferred"]:
            err.append({"id": self._id("batch_req"), "custom_id": line["custom_id"], "response": None,
                        "error": {"code": "batch_expired", "message": "This request could not be executed before the completion window expired."}})
        self.rng.shuffle(out)
        self.rng.shuffle(err)                                         # output order may not match input order
        usages = [r["response"]["body"]["usage"] for r in out]
        tin, tout = sum(u["input_tokens"] for u in usages), sum(u["output_tokens"] for u in usages)
        b["usage"] = {"input_tokens": tin, "output_tokens": tout, "total_tokens": tin + tout,
                      "input_tokens_details": {"cached_tokens": 0}, "output_tokens_details": {"reasoning_tokens": 32 * len(usages)}}
        b["output_file_id"], b["error_file_id"] = self._store(out), self._store(err)
        b["request_counts"]["failed"] += len(b["_deferred"])
        b["status"] = "expired" if b["_deferred"] else "completed"    # the window ran out before every line ran


print("toolkit ready: FakeClaudeBatches, FakeOpenAIBatches, claude_responder, openai_responder")

In [ ]:
def claude_followup_params(doc: Doc, failed: dict, issues: list[Issue], *, system: str = SYSTEM_FEW_SHOT,
                           model: str = CLAUDE_MODEL, pressure: bool = False) -> dict:
    """Document -> failed extraction -> specific errors, as a pre-built Messages conversation."""
    return {"model": model, "max_tokens": 1024, "system": system,
            "output_config": {"format": {"type": "json_schema", "schema": CLAUDE_SCHEMA}},
            "messages": [{"role": "user", "content": doc_block(doc.doc_id, doc.text)},
                         {"role": "assistant", "content": json.dumps(failed)},
                         {"role": "user", "content": feedback_text(issues, pressure)}]}


def openai_followup_body(doc: Doc, failed: dict, issues: list[Issue], *, system: str = SYSTEM_FEW_SHOT,
                         model: str = OPENAI_MODEL) -> dict:
    return {"model": model, "instructions": system, "max_output_tokens": 2000, "reasoning": {"effort": "low"},
            "text": {"format": {"type": "json_schema", "name": "document_extraction", "schema": OPENAI_SCHEMA, "strict": True}},
            "input": [{"role": "user", "content": doc_block(doc.doc_id, doc.text)},
                      {"role": "assistant", "content": json.dumps(failed)},
                      {"role": "user", "content": feedback_text(issues)}]}


fc, fo = FakeClaudeBatches(), FakeOpenAIBatches()
retry_log = []                                     # what retry fixed, and what it could not
for d in SAMPLE:
    first = simulated_extraction(doc_block(d.doc_id, d.text), SYSTEM_PLAIN, CLAUDE_SCHEMA)
    issues = validate_extraction(first)
    if not issues:
        continue
    msg = fc.client.messages.create(**claude_followup_params(d, first, issues, system=SYSTEM_PLAIN))
    second = json.loads(msg.content[0].text)
    left = {(i.field, i.code) for i in validate_extraction(second)}
    for i in issues:
        source_has_it = not (i.code == "missing_total" and d.truth["stated_total_cents"] is None)
        retry_log.append((i.code, (i.field, i.code) not in left, source_has_it))

table = defaultdict(lambda: [0, 0])
for code, fixed, source_has_it in retry_log:
    key = (code, "value in source" if source_has_it else "value ABSENT from source")
    table[key][0] += fixed
    table[key][1] += 1
print(f"{'issue':22} {'root cause':26} fixed by one retry")
for (code, cause), (fixed, n) in sorted(table.items()):
    print(f"{code:22} {cause:26} {fixed}/{n}")

r = fo.client.responses.create(**openai_followup_body(SAMPLE[0], {"doc_type": "invoice"}, [Issue("x", "schema", "demo", True)]))
print("\nOpenAI follow-up through the real SDK:", r.status, "| output_text parses:", bool(json.loads(r.output_text)))

> **Observe:** format and structure issues (`date_format`, `sum_mismatch` from a misread decimal comma, `other_without_detail`, missed totals that **are** in the text) are fixed by one retry with specific feedback. `missing_total` on a torn receipt is not: the information is absent, and a retry returns `null` again, which is the correct, honest answer. Track the two populations separately; the second one belongs in the human queue after **one** attempt, not three.

In [ ]:
# The anti-pattern: "fix" the absent total by insisting on a value
d = next(x for x in SAMPLE if "absent_total" in x.tags)
first = simulated_extraction(doc_block(d.doc_id, d.text), SYSTEM_FEW_SHOT, CLAUDE_SCHEMA)
issues = validate_extraction(first)
pressured = json.loads(fc.client.messages.create(**claude_followup_params(d, first, issues, pressure=True)).content[0].text)
print("label:", d.truth["stated_total_cents"], "| after a neutral retry: null kept | after a 'must not be null' retry:",
      pressured["stated_total_cents"], "(fabricated: the sum of the readable lines)")

> **Observe:** the pressured retry "succeeds": the output is now schema-valid and passes `missing_total`. It is also wrong, and nothing downstream will notice. That is why the feedback text says *keep it null if it is truly not in the document*, and why the pipeline caps this issue at one retry and then routes to a person.

> **Try it:** change `TOTAL_REQUIRED_FOR` to exclude receipts. What happens to the torn receipts now, and is that acceptable for accounts payable?

## 3.5 Batch processing on both stacks

The pipeline submits the 100 documents as **one batch**, waits for it, normalizes each result by `custom_id`, and triages: valid → accept; validation issues → retry with feedback; `expired` and transient server errors → resubmit **as is**; "prompt too long" → **chunk** (`doc-0042-c0`, `-c1`, ...) and resubmit only those; truncated output → raise the output cap; refusals and unfixable issues → human. Only failed `custom_id`s are ever resubmitted, and every document ends in exactly one terminal state. The vendor-specific code lives in two small adapters; everything else is shared. [Theory](README.md#35-batch-processing-on-both-stacks)

In [ ]:
@dataclass
class ExtractionRequest:
    """Vendor-neutral request: one document (or chunk), plus feedback turns for a retry."""
    custom_id: str
    document: str                                   # already wrapped in <document id=...>
    failed: dict | None = None                      # the previous extraction, for retry-with-feedback
    feedback: str | None = None
    max_tokens: int = 1024


@dataclass
class Outcome:
    custom_id: str
    status: str                                     # ok | errored | expired | canceled | truncated | refused | invalid_output
    data: dict | None = None
    error_type: str | None = None
    error_message: str | None = None
    input_tokens: int = 0
    output_tokens: int = 0


class ClaudeBatchBackend:
    """Message Batches API adapter. mode='json' uses output_config.format; mode='tool' uses a strict extraction tool."""
    name = "claude"

    def __init__(self, client: anthropic.Anthropic, *, model: str = CLAUDE_MODEL, system: str = SYSTEM_FEW_SHOT,
                 mode: Literal["json", "tool"] = "json", tool_choice: dict | None = None,
                 sleep: Callable[[float], None] = time.sleep, poll_seconds: float = 60, max_wait_s: float = 25 * 3600):
        self.client, self.model, self.system, self.mode = client, model, system, mode
        self.tool_choice = tool_choice or {"type": "auto"}
        self.sleep, self.poll_seconds, self.max_wait_s = sleep, poll_seconds, max_wait_s
        self.tool = {"name": "record_extraction", "strict": True, "input_schema": CLAUDE_SCHEMA,
                     "description": "Record the accounts-payable fields extracted from ONE document. Call it exactly once."}

    def params(self, r: ExtractionRequest) -> MessageCreateParamsNonStreaming:
        messages: list[dict] = [{"role": "user", "content": r.document}]
        p: dict = {"model": self.model, "max_tokens": r.max_tokens, "system": self.system}
        if self.mode == "json":
            p["output_config"] = {"format": {"type": "json_schema", "schema": CLAUDE_SCHEMA}}
            if r.failed is not None:
                messages += [{"role": "assistant", "content": json.dumps(r.failed)}, {"role": "user", "content": r.feedback}]
        else:
            p["tools"], p["tool_choice"] = [self.tool], self.tool_choice
            if r.failed is not None:                  # tool feedback = a tool_result with is_error
                tid = "toolu_retry_" + r.custom_id.replace("-", "_")[:40]
                messages += [{"role": "assistant", "content": [{"type": "tool_use", "id": tid, "name": self.tool["name"], "input": r.failed}]},
                             {"role": "user", "content": [{"type": "tool_result", "tool_use_id": tid, "is_error": True, "content": r.feedback}]}]
        p["messages"] = messages
        return MessageCreateParamsNonStreaming(**p)

    def submit(self, requests: list[ExtractionRequest]) -> str:
        batch = self.client.messages.batches.create(requests=[Request(custom_id=r.custom_id, params=self.params(r)) for r in requests])
        return batch.id

    def wait(self, batch_id: str) -> str:
        waited, delay = 0.0, self.poll_seconds
        while (b := self.client.messages.batches.retrieve(batch_id)).processing_status != "ended":
            if waited >= self.max_wait_s:
                return "timeout"
            self.sleep(delay)
            waited += delay
            delay = min(delay * 2, 600)                # back off: status calls count against batch RPM
        return "ended"

    def _normalize_message(self, cid: str, msg) -> Outcome:
        o = Outcome(cid, "ok", input_tokens=msg.usage.input_tokens, output_tokens=msg.usage.output_tokens)
        if msg.stop_reason in ("max_tokens", "model_context_window_exceeded"):
            o.status = "truncated"
        elif msg.stop_reason == "refusal":
            o.status = "refused"
        elif self.mode == "tool":
            call = next((b for b in msg.content if b.type == "tool_use" and b.name == self.tool["name"]), None)
            o.status, o.data = ("ok", call.input) if call else ("invalid_output", None)
        else:
            try:
                o.data = json.loads("".join(b.text for b in msg.content if b.type == "text"))
            except json.JSONDecodeError:
                o.status = "invalid_output"
        return o

    def results(self, batch_id: str) -> list[Outcome]:
        outcomes = []
        for item in self.client.messages.batches.results(batch_id):      # streamed, typed, in ANY order
            r = item.result
            if r.type == "succeeded":
                outcomes.append(self._normalize_message(item.custom_id, r.message))
            elif r.type == "errored":
                outcomes.append(Outcome(item.custom_id, "errored", error_type=r.error.error.type, error_message=r.error.error.message))
            else:                                       # expired / canceled: never ran, not billed
                outcomes.append(Outcome(item.custom_id, r.type))
        return outcomes

    def call_sync(self, r: ExtractionRequest) -> Outcome:
        return self._normalize_message(r.custom_id, self.client.messages.create(**self.params(r)))

    API_ERRORS = (anthropic.APIError,)                  # what call_sync may raise once the SDK's own retries are spent

    def sync_error(self, cid: str, e: Exception) -> Outcome:
        """Turn an SDK exception into an Outcome, so one failed request cannot abort the whole round."""
        if isinstance(e, anthropic.APIStatusError):
            err = e.body.get("error", {}) if isinstance(e.body, dict) else {}
            return Outcome(cid, "errored", error_type=err.get("type") or f"http_{e.status_code}", error_message=err.get("message") or str(e))
        if isinstance(e, anthropic.APIConnectionError):                   # includes APITimeoutError: transient
            return Outcome(cid, "errored", error_type="connection_error", error_message=str(e))
        return Outcome(cid, "errored", error_type=type(e).__name__, error_message=str(e))   # e.g. a malformed response: a person looks


class OpenAIBatchBackend:
    """Batch API adapter: JSONL file -> files.create(purpose='batch') -> batches.create('/v1/responses') -> output/error files."""
    name = "openai"
    TERMINAL = {"completed", "failed", "expired", "cancelled"}

    def __init__(self, client: OpenAI, *, model: str = OPENAI_MODEL, system: str = SYSTEM_FEW_SHOT,
                 sleep: Callable[[float], None] = time.sleep, poll_seconds: float = 60, max_wait_s: float = 25 * 3600):
        self.client, self.model, self.system = client, model, system
        self.sleep, self.poll_seconds, self.max_wait_s = sleep, poll_seconds, max_wait_s

    def body(self, r: ExtractionRequest) -> dict:
        items: list[dict] = [{"role": "user", "content": r.document}]
        if r.failed is not None:
            items += [{"role": "assistant", "content": json.dumps(r.failed)}, {"role": "user", "content": r.feedback}]
        return {"model": self.model, "instructions": self.system, "input": items, "reasoning": {"effort": "low"},
                "max_output_tokens": r.max_tokens * 2,     # headroom for reasoning tokens, which count as output
                "text": {"format": {"type": "json_schema", "name": "document_extraction", "schema": OPENAI_SCHEMA, "strict": True}}}

    def submit(self, requests: list[ExtractionRequest]) -> str:
        path = WORKDIR / f"batch_{next(_ids):05d}.jsonl"
        path.write_text("".join(json.dumps({"custom_id": r.custom_id, "method": "POST", "url": "/v1/responses",
                                            "body": self.body(r)}) + "\n" for r in requests), encoding="utf-8")
        with open(path, "rb") as f:
            file = self.client.files.create(file=f, purpose="batch")
        batch = self.client.batches.create(input_file_id=file.id, endpoint="/v1/responses", completion_window="24h",
                                           metadata={"job": "cap03-extraction", "prompt_version": "fewshot-v1"})
        return batch.id

    def wait(self, batch_id: str) -> str:
        waited = 0.0
        while (b := self.client.batches.retrieve(batch_id)).status not in self.TERMINAL:
            if waited >= self.max_wait_s:
                return "timeout"
            self.sleep(self.poll_seconds)
            waited += self.poll_seconds
        return b.status

    def _read(self, file_id: str | None) -> list[dict]:
        if not file_id:
            return []
        return [json.loads(x) for x in self.client.files.content(file_id).text.splitlines() if x.strip()]

    def _normalize_body(self, cid: str, body: dict) -> Outcome:
        u = body.get("usage") or {}
        o = Outcome(cid, "ok", input_tokens=u.get("input_tokens", 0), output_tokens=u.get("output_tokens", 0))
        if body.get("status") != "completed":
            o.status = "truncated" if (body.get("incomplete_details") or {}).get("reason") == "max_output_tokens" else "errored"
            return o
        parts = [p for it in body.get("output", []) if it.get("type") == "message" for p in it.get("content", [])]
        if any(p.get("type") == "refusal" for p in parts):
            o.status = "refused"
            return o
        try:
            o.data = json.loads("".join(p["text"] for p in parts if p.get("type") == "output_text"))
        except json.JSONDecodeError:
            o.status = "invalid_output"
        return o

    def results(self, batch_id: str) -> list[Outcome]:
        b = self.client.batches.retrieve(batch_id)
        if b.status == "failed":                       # the whole file was rejected at validation: nothing ran
            raise RuntimeError(f"batch failed validation: {[e.message for e in b.errors.data]}")
        outcomes = []
        for line in self._read(b.output_file_id) + self._read(b.error_file_id):
            cid, resp = line["custom_id"], line.get("response")
            if line.get("error"):
                code = line["error"].get("code")
                outcomes.append(Outcome(cid, "expired" if code == "batch_expired" else "errored", error_type=code))
            elif resp["status_code"] != 200:
                err = resp["body"].get("error", {})
                outcomes.append(Outcome(cid, "errored", error_type=err.get("code") or err.get("type") or f"http_{resp['status_code']}",
                                        error_message=err.get("message")))
            else:
                outcomes.append(self._normalize_body(cid, resp["body"]))
        return outcomes

    def call_sync(self, r: ExtractionRequest) -> Outcome:
        return self._normalize_body(r.custom_id, self.client.responses.create(**self.body(r)).model_dump(mode="json"))

    API_ERRORS = (openai.APIError,)

    def sync_error(self, cid: str, e: Exception) -> Outcome:
        if isinstance(e, openai.APIStatusError):
            return Outcome(cid, "errored", error_type=e.code or e.type or f"http_{e.status_code}", error_message=e.message)
        if isinstance(e, openai.APIConnectionError):
            return Outcome(cid, "errored", error_type="connection_error", error_message=str(e))
        return Outcome(cid, "errored", error_type=type(e).__name__, error_message=str(e))


print("adapters ready: ClaudeBatchBackend, OpenAIBatchBackend")

### One batch through each adapter, step by step (offline)

In [ ]:
def first_pass_requests(docs: list[Doc], max_tokens: int = 1024) -> list[ExtractionRequest]:
    return [ExtractionRequest(d.doc_id, doc_block(d.doc_id, d.text), max_tokens=max_tokens) for d in docs]


fake_claude = FakeClaudeBatches(duration_minutes=55)
claude = ClaudeBatchBackend(fake_claude.client, sleep=fake_claude.sleep)
bid = claude.submit(first_pass_requests(CORPUS))
print("Claude batch:", bid, "| wait ->", claude.wait(bid), "| virtual time:", fake_claude.now - datetime(2026, 10, 3, 18, 0, tzinfo=timezone.utc))
c_out = claude.results(bid)
print("statuses:", dict(Counter(o.status for o in c_out)))
print("errored:", dict(Counter(o.error_type for o in c_out if o.status == "errored")))
print("first 5 custom_ids in the results:", [o.custom_id for o in c_out[:5]], "(not request order)")

fake_openai = FakeOpenAIBatches()
oai = OpenAIBatchBackend(fake_openai.client, sleep=fake_openai.sleep)
bid = oai.submit(first_pass_requests(CORPUS))
print("\nOpenAI batch:", bid, "| wait ->", oai.wait(bid))
o_out = oai.results(bid)
print("statuses:", dict(Counter(o.status for o in o_out)))
print("errored:", dict(Counter(o.error_type for o in o_out if o.status == "errored")))
print("HTTP calls the OpenAI SDK made:", Counter(x.split(" ")[0] + " " + re.sub(r"_\d+", "_*", x.split(" ")[1]) for x in fake_openai.log))

> **Observe:** the same 100 documents produce the same *kinds* of failure on both stacks, reported differently. Claude returns one stream with four result types (`errored` with `invalid_request_error` for the oversized documents, `overloaded_error` for the transient ones, `expired`); OpenAI ends the batch as `expired` (two lines never ran) and splits results across an output file and an error file (`context_length_exceeded`, which is the fake's illustrative code, a `500`, and the documented `batch_expired`). The OpenAI batch status is `expired` even though 98 lines ran: read the files, not just the status. The adapters normalize both into `Outcome`s keyed by `custom_id`.

### The pipeline: triage, resubmit only failures, chunk, retry with feedback

In [ ]:
TRANSIENT_ERRORS = {"overloaded_error", "api_error", "server_error", "rate_limit_error", "connection_error",
                    "http_500", "http_502", "http_503", None}
TOO_LONG_ERRORS = ("prompt is too long", "context_length_exceeded")   # Claude: documented 400 message; OpenAI: the fake's illustrative code


def chunk_document(doc: Doc, max_tokens: int = 2_500) -> list[str]:
    """Split on line boundaries so each chunk (plus the prompt) fits the context window."""
    chunks, current = [], []
    for line in doc.text.splitlines():
        if current and approx_tokens("\n".join(current + [line])) > max_tokens:
            chunks.append("\n".join(current))
            current = []
        current.append(line)
    return chunks + (["\n".join(current)] if current else [])


def merge_chunks(parts: list[dict]) -> dict:
    """First non-null value per field (header facts live in chunk 0); line items concatenated."""
    merged = copy.deepcopy(parts[0])
    for p in parts[1:]:
        for k, v in p.items():
            if k == "line_items":
                merged["line_items"] = merged["line_items"] + v
            elif merged.get(k) is None and v is not None:
                merged[k] = v
    merged["conflict_detected"] = any(p.get("conflict_detected") for p in parts)
    return merged


@dataclass
class LedgerEntry:
    doc_id: str
    state: str = "pending"                  # pending -> accepted | human_review   (exactly one terminal state)
    reason: str = ""
    rounds: int = 0
    data: dict | None = None
    history: list[str] = field(default_factory=list)
    issues_retried: list[str] = field(default_factory=list)


class BatchStillRunning(Exception):
    """wait() gave up before the batch reached a terminal status. The batch keeps running on the provider's side;
    its ID is in the manifest, and ExtractionPipeline.resume(...).run() re-attaches to it later."""
    def __init__(self, batch_id: str, manifest_path: Path):
        super().__init__(f"batch {batch_id} has not ended yet; state saved in {manifest_path.name}")
        self.batch_id, self.manifest_path = batch_id, manifest_path


class ExtractionPipeline:
    def __init__(self, backend, docs: list[Doc], *, max_tokens: int = 1024, max_rounds: int = 3,
                 retry_via: Literal["batch", "sync"] = "batch", batch_cycles: int | None = None, preflight: bool = False):
        self.backend, self.docs = backend, {d.doc_id: d for d in docs}
        self.max_tokens, self.max_rounds, self.retry_via, self.preflight = max_tokens, max_rounds, retry_via, preflight
        # Rounds 0 .. batch_cycles-1 go through the batch API; later rounds go synchronous. retry_via is the shorthand:
        # "sync" = only the first pass is a batch; "batch" = every round (up to max_rounds + 1 submissions) is a batch.
        self.batch_cycles = batch_cycles if batch_cycles is not None else (1 if retry_via == "sync" else max_rounds + 1)
        self.ledger = {d.doc_id: LedgerEntry(d.doc_id) for d in docs}
        self.chunk_parts: dict[str, dict[int, dict]] = defaultdict(dict)
        self.chunk_count: dict[str, int] = {}
        self.usage = defaultdict(lambda: [0, 0])  # round -> [input, output] tokens
        self.round_mode: dict[int, str] = {}      # round -> "batch" | "sync" (for pricing)
        self.batches: list[tuple[int, str, int]] = []   # (round, batch_id, n requests)
        self.pending: list[ExtractionRequest] = []     # the requests of the round in progress (full payloads)
        self.next_round, self.in_flight, self.started = 0, None, False   # in_flight = {"round", "batch_id"} while waiting
        self.manifest_path = WORKDIR / f"manifest_{backend.name}_{next(_ids):05d}.json"   # crash-safe state on disk

    def save_manifest(self) -> None:
        """Persist EVERYTHING a new process needs to continue: the in-flight batch ID, the full pending requests
        (retry feedback, chunk texts, raised output caps), chunk results, usage and the whole ledger."""
        state = {"backend": self.backend.name, "next_round": self.next_round, "in_flight": self.in_flight,
                 "pending": [asdict(r) for r in self.pending],
                 "batches": [{"round": r, "batch_id": b, "n": n} for r, b, n in self.batches],
                 "chunk_count": self.chunk_count,
                 "chunk_parts": {d: {str(i): v for i, v in parts.items()} for d, parts in self.chunk_parts.items()},
                 "usage": {str(r): u for r, u in self.usage.items()}, "round_mode": {str(r): m for r, m in self.round_mode.items()},
                 "ledger": {k: asdict(e) for k, e in self.ledger.items()}}
        tmp = self.manifest_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(state, indent=1), encoding="utf-8")
        tmp.replace(self.manifest_path)                                # atomic rename: never a half-written file

    @classmethod
    def resume(cls, manifest_path: Path, backend, docs: list[Doc], **kwargs) -> "ExtractionPipeline":
        """Rebuild a pipeline from its manifest (for example in a new process after a crash or a wait timeout)."""
        state = json.loads(Path(manifest_path).read_text())
        assert state["backend"] == backend.name, "resume with the same vendor's backend"
        pipe = cls(backend, docs, **kwargs)
        pipe.manifest_path, pipe.started = Path(manifest_path), True
        pipe.next_round, pipe.in_flight = state["next_round"], state["in_flight"]
        pipe.pending = [ExtractionRequest(**r) for r in state["pending"]]
        pipe.batches = [(b["round"], b["batch_id"], b["n"]) for b in state["batches"]]
        pipe.chunk_count = state["chunk_count"]
        pipe.chunk_parts = defaultdict(dict, {d: {int(i): v for i, v in p.items()} for d, p in state["chunk_parts"].items()})
        pipe.usage = defaultdict(lambda: [0, 0], {int(r): u for r, u in state["usage"].items()})
        pipe.round_mode = {int(r): m for r, m in state["round_mode"].items()}
        pipe.ledger = {k: LedgerEntry(**v) for k, v in state["ledger"].items()}
        return pipe

    # ---- request building ------------------------------------------------------------
    def _chunk_requests(self, doc: Doc) -> list[ExtractionRequest]:
        parts = chunk_document(doc)
        self.chunk_count[doc.doc_id] = len(parts)
        return [ExtractionRequest(f"{doc.doc_id}-c{i}", doc_block(doc.doc_id, p, chunk=i), max_tokens=self.max_tokens)
                for i, p in enumerate(parts)]

    def initial_requests(self) -> list[ExtractionRequest]:
        reqs = []
        for d in self.docs.values():
            too_long = approx_tokens(SYSTEM_FEW_SHOT + d.text) > FAKE_CONTEXT_TOKENS   # live: client.messages.count_tokens
            reqs += self._chunk_requests(d) if (self.preflight and too_long) else \
                [ExtractionRequest(d.doc_id, doc_block(d.doc_id, d.text), max_tokens=self.max_tokens)]
        return reqs

    # ---- triage ----------------------------------------------------------------------
    def _finish(self, doc_id: str, state: str, reason: str, data: dict | None = None) -> None:
        e = self.ledger[doc_id]
        e.state, e.reason, e.data = state, reason, data
        e.history.append(f"r{e.rounds}:{state}:{reason}")

    def _check(self, doc_id: str, data: dict, rnd: int, req_by_id: dict) -> list[ExtractionRequest]:
        e = self.ledger[doc_id]
        issues = validate_extraction(data)
        if not issues:
            if data.get("conflict_detected"):
                self._finish(doc_id, "human_review", "source_conflict: printed total differs from line items", data)
            else:
                self._finish(doc_id, "accepted", "first_pass" if rnd == 0 else f"after_round_{rnd}", data)
            return []
        if doc_id in self.chunk_count:                 # a chunked document: a full-text retry would not fit either
            self._finish(doc_id, "human_review", "chunked document failed validation: " + ",".join(sorted({i.code for i in issues})), data)
            return []
        retryable = [i for i in issues if i.retryable and not (i.code == "missing_total" and "missing_total" in e.issues_retried)]
        if not retryable or rnd + 1 >= self.max_rounds:
            self._finish(doc_id, "human_review", "unresolved: " + ",".join(sorted({i.code for i in issues})), data)
            return []
        e.issues_retried += [i.code for i in retryable]
        doc = self.docs[doc_id]
        return [ExtractionRequest(doc_id, doc_block(doc.doc_id, doc.text), failed=data, feedback=feedback_text(retryable),
                                  max_tokens=self.max_tokens)]

    def triage(self, outcomes: list[Outcome], rnd: int, req_by_id: dict) -> list[ExtractionRequest]:
        nxt: list[ExtractionRequest] = []
        for o in outcomes:
            self.usage[rnd][0] += o.input_tokens
            self.usage[rnd][1] += o.output_tokens
            doc_id = base_id(o.custom_id)
            e = self.ledger[doc_id]
            e.rounds = rnd
            e.history.append(f"r{rnd}:{o.custom_id}:{o.status}" + (f"({o.error_type})" if o.error_type else ""))
            req = req_by_id[o.custom_id]
            if o.status == "ok" and o.custom_id != doc_id:          # one chunk of a chunked document
                self.chunk_parts[doc_id][int(o.custom_id.rsplit("-c", 1)[1])] = o.data
                if len(self.chunk_parts[doc_id]) == self.chunk_count[doc_id]:
                    parts = [self.chunk_parts[doc_id][i] for i in range(self.chunk_count[doc_id])]
                    nxt += self._check(doc_id, merge_chunks(parts), rnd, req_by_id)
            elif o.status == "ok":
                nxt += self._check(doc_id, o.data, rnd, req_by_id)
            elif o.status in ("expired", "canceled") or (o.status == "errored" and o.error_type in TRANSIENT_ERRORS):
                nxt.append(req)                                        # never ran / transient: resubmit AS IS
            elif o.status == "errored" and any(s in f"{o.error_type} {o.error_message}" for s in TOO_LONG_ERRORS):
                if o.custom_id != doc_id:
                    self._finish(doc_id, "human_review", "chunk still too long")
                else:
                    nxt += self._chunk_requests(self.docs[doc_id])     # MODIFY: split into chunks, new -cN ids
            elif o.status == "truncated" and req.max_tokens < 8 * self.max_tokens:
                nxt.append(ExtractionRequest(req.custom_id, req.document, req.failed, req.feedback, req.max_tokens * 2))
            else:                                                      # refused, invalid request, invalid output
                self._finish(doc_id, "human_review", f"{o.status}:{o.error_type or ''}")
        return nxt

    # ---- run ---------------------------------------------------------------------------
    def _call_sync(self, r: ExtractionRequest) -> Outcome:
        try:
            return self.backend.call_sync(r)
        except self.backend.API_ERRORS as e:                            # one failed request must not abort the round
            return self.backend.sync_error(r.custom_id, e)

    def run(self, verbose: bool = True) -> dict[str, LedgerEntry]:
        """Run (or continue) the rounds. Safe to call again on a resumed pipeline: an in-flight batch is re-attached
        by ID, never resubmitted, and results are read only after the batch reaches a terminal status."""
        if not self.started:
            self.pending, self.next_round, self.started = self.initial_requests(), 0, True
            self.save_manifest()
        while self.pending and self.next_round <= self.max_rounds:
            rnd = self.next_round
            req_by_id = {r.custom_id: r for r in self.pending}
            if rnd >= self.batch_cycles:                                 # no SLA room for another batch cycle
                outcomes = [self._call_sync(r) for r in self.pending]
                label, self.round_mode[rnd] = "sync", "sync"
            else:
                if self.in_flight is None:
                    bid = self.backend.submit(self.pending)
                    self.batches.append((rnd, bid, len(self.pending)))
                    self.in_flight = {"round": rnd, "batch_id": bid}
                    self.save_manifest()                                # persist the batch ID BEFORE the long wait
                bid = self.in_flight["batch_id"]                         # after a restart: the saved ID, re-attached
                status = self.backend.wait(bid)
                if status == "timeout":                                  # still running: keep the ID, do NOT resubmit
                    self.save_manifest()
                    raise BatchStillRunning(bid, self.manifest_path)
                outcomes = self.backend.results(bid)                     # only now: the batch is in a terminal status
                label, self.round_mode[rnd] = f"batch {bid} ({status})", "batch"
            seen = {o.custom_id for o in outcomes}
            missing = set(req_by_id) - seen                             # reconciliation: nothing silently lost
            outcomes += [Outcome(cid, "expired") for cid in missing]
            if verbose:
                print(f"round {rnd}: {len(self.pending):3} requests via {label}: {dict(Counter(o.status for o in outcomes))}")
            self.pending = self.triage(outcomes, rnd, req_by_id)
            self.next_round, self.in_flight = rnd + 1, None
            self.save_manifest()                                        # checkpoint: a crash now repeats nothing
        for e in self.ledger.values():
            if e.state == "pending":
                self._finish(e.doc_id, "human_review", "attempts exhausted")
        self.save_manifest()
        return self.ledger


def coverage_report(ledger: dict[str, LedgerEntry]) -> Counter:
    states = Counter(e.state if e.state != "accepted" else f"accepted ({e.reason})" for e in ledger.values())
    assert sum(states.values()) == len(ledger), "every document must end in exactly one terminal state"
    return states


claude_pipe = ExtractionPipeline(ClaudeBatchBackend((fc := FakeClaudeBatches()).client, sleep=fc.sleep), CORPUS)
claude_ledger = claude_pipe.run()
print(dict(coverage_report(claude_ledger)))
print("human review reasons:", dict(Counter(e.reason.split(":")[0] for e in claude_ledger.values() if e.state == "human_review")))
print("\nhistory of one chunked document:", next(e.history for e in claude_ledger.values() if any("-c0" in h for h in e.history)))

In [ ]:
openai_pipe = ExtractionPipeline(OpenAIBatchBackend((fo := FakeOpenAIBatches()).client, sleep=fo.sleep), CORPUS)
openai_ledger = openai_pipe.run()
print(dict(coverage_report(openai_ledger)))
same = sum(claude_ledger[k].state == openai_ledger[k].state for k in claude_ledger)
print(f"documents with the same terminal state on both stacks: {same}/100")

> **Observe:**
> - Round 1 contains **only** the failures from round 0: the transient and expired documents unchanged, the oversized documents as chunks (`-c0`, `-c1`, ...), the truncated ones with a doubled output cap, and the validation failures with feedback. Nothing that succeeded is ever resubmitted.
> - The source-conflict documents go to people with both numbers; the torn receipts go to people after one honest retry.
> - The manifest is written right after each submit (before the long wait) and after each round. It holds the full pending requests, not just their IDs, so the next section can kill the process mid-wait and continue in a new one.
> - Both stacks end with the same terminal states because the simulated model is the same; what differs is the plumbing (inline requests vs a JSONL file, one results stream vs two files).

> **Try it:** run `ExtractionPipeline(..., preflight=True)` on Claude. Pre-flight token counting chunks the oversized documents **before** submission. How many requests does round 1 contain now?

### Crash, timeout and network-error recovery (offline)

Three failures a long-running batch job meets in production, each injected on purpose: the process dies while waiting; the wait gives up before the batch ends; a synchronous retry hits a network error. In each case nothing may be submitted twice and every document must still end in exactly one terminal state.

In [ ]:
class ProcessKilled(Exception):
    pass


# 1. The process dies during the wait. A NEW pipeline object (a new process) resumes from the manifest file alone.
fk = FakeClaudeBatches()
dying = ClaudeBatchBackend(fk.client, sleep=fk.sleep)


def killed_during_wait(batch_id: str) -> str:
    raise ProcessKilled(f"killed while waiting for {batch_id}")


dying.wait = killed_during_wait
first = ExtractionPipeline(dying, CORPUS)
try:
    first.run(verbose=False)
except ProcessKilled as e:
    print("process 1:", e, "| batches created so far:", len(fk.batches))
saved = json.loads(first.manifest_path.read_text())
print(f"manifest: in_flight={saved['in_flight']}, {len(saved['pending'])} full pending requests, round {saved['next_round']}")
second = ExtractionPipeline.resume(first.manifest_path, ClaudeBatchBackend(fk.client, sleep=fk.sleep), CORPUS)
second.run(verbose=False)
print(f"process 2: re-attached to {second.batches[0][1]} | batches created in total: {len(fk.batches)} "
      f"(uninterrupted run: {len(claude_pipe.batches)}) | same terminal states: "
      f"{all(second.ledger[k].state == claude_ledger[k].state for k in claude_ledger)}")

# 2. The wait times out (a 10-hour batch, a 1-hour wait). Results are NOT requested early, nothing is resubmitted.
slow = FakeClaudeBatches(duration_minutes=600)
docs10 = CORPUS[:10]
impatient = ExtractionPipeline(ClaudeBatchBackend(slow.client, sleep=slow.sleep, max_wait_s=3600), docs10)
try:
    impatient.run(verbose=False)
except BatchStillRunning as e:
    print("\ntimeout:", e, "| batches created:", len(slow.batches))
later = ExtractionPipeline.resume(impatient.manifest_path, ClaudeBatchBackend(slow.client, sleep=slow.sleep), docs10)
later.run(verbose=False)
print(f"resumed later: first-pass batch {later.batches[0][1]} submitted once | batches created in total: {len(slow.batches)} | "
      f"{dict(coverage_report(later.ledger))}")

# 3. A synchronous retry hits a connection error (the SDK's own retries are already spent: max_retries=0 here).
fn = FakeClaudeBatches()
drops = {"left": 1}


def flaky_transport(request: httpx2.Request) -> httpx2.Response:
    if request.url.path == "/v1/messages" and drops["left"]:
        drops["left"] -= 1
        raise httpx2.ConnectError("connection reset by peer")
    return fn._handle(request)


flaky = anthropic.Anthropic(api_key="offline", max_retries=0,
                            http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(flaky_transport)))
transient_docs = [d for d in CORPUS if "transient" in d.tags]
net = ExtractionPipeline(ClaudeBatchBackend(flaky, sleep=fn.sleep), transient_docs, retry_via="sync")
net.run()
print({k: (e.state, e.history[-1]) for k, e in net.ledger.items()})

> **Observe:**
> - **Crash:** the restarted process submits nothing for the round that was in flight. It re-attaches to the saved batch ID, and the run ends with the same batch count and the same terminal states as the uninterrupted run. This works because the manifest stores the **complete** pending requests (retry feedback, chunk texts, raised output caps), the chunk results and the ledger, not only the IDs.
> - **Timeout:** `wait()` giving up is not a result. Calling `results()` then would fail on Claude (no `results_url` until the batch ends) and return no output file on OpenAI, which a naive reconciliation would misread as "expired" and resubmit. The pipeline raises `BatchStillRunning` with the ID saved; you resume later.
> - **Network error:** the failed synchronous call becomes an `errored` outcome (`connection_error`, transient) for that one document, is retried in the next round, and the rest of the round is unaffected. The number of rounds bounds the retries; a document still failing after the last round ends in `human_review` with "attempts exhausted".

### Claude: strict tool instead of JSON outputs, and the forced `tool_choice` trap (offline)

In [ ]:
tool_backend = ClaudeBatchBackend((fct := FakeClaudeBatches()).client, sleep=fct.sleep, mode="tool")
sample10 = CORPUS[20:30]
bid = tool_backend.submit(first_pass_requests(sample10))
tool_backend.wait(bid)
print("strict tool + tool_choice auto on", CLAUDE_MODEL, "->", dict(Counter(o.status for o in tool_backend.results(bid))))

forced = ClaudeBatchBackend((fcf := FakeClaudeBatches()).client, sleep=fcf.sleep, mode="tool",
                            tool_choice={"type": "tool", "name": "record_extraction"})
bid = forced.submit(first_pass_requests(sample10))
forced.wait(bid)
res = forced.results(bid)
print("forced tool_choice on", CLAUDE_MODEL, "->", dict(Counter(o.status for o in res)), "|", res[0].error_message)
forced_haiku = ClaudeBatchBackend((fch := FakeClaudeBatches()).client, sleep=fch.sleep, mode="tool", model="claude-haiku-4-5",
                                  tool_choice={"type": "tool", "name": "record_extraction"})
bid = forced_haiku.submit(first_pass_requests(sample10))
forced_haiku.wait(bid)
print("forced tool_choice on claude-haiku-4-5 ->", dict(Counter(o.status for o in forced_haiku.results(bid))))

> **Observe:** a forced `tool_choice` is the textbook way to guarantee a structured call, and it still works on models such as Claude Haiku 4.5. On Claude Sonnet 5.5 (and Opus 5.5, Fable 5.1) it is rejected, inside batches too, and the rejection only surfaces when the batch **ends**: one more reason to dry-run the exact request shape synchronously before a large submission. On those models, use `output_config.format` (the default here) or a strict tool with `tool_choice: auto` plus a code check that the call happened (`invalid_output` otherwise).

## 3.6 SLA math

The client's SLA: an extraction in the AP system **30 hours** after a document arrives. Batches have **no latency SLA**: plan with the 24-hour bound, not with the typical hour. The worst case for a document that just missed a submission is `W + 24 h + B`. Retries change the arithmetic: every further **batch** cycle adds up to another 24 hours, so the pipeline allows only as many batch rounds as the SLA absorbs (`batch_cycles`) and sends the remaining rounds synchronously. [Theory](README.md#36-sla-math)

In [ ]:
def submission_window(sla_h: float, processing_h: float = 24, buffer_h: float = 2, retry_mode: str = "sync",
                      batch_cycles: int | None = None) -> float:
    """Longest interval between submissions so that the worst case still meets the SLA.
    batch_cycles = how many batch submissions one document may need; retry_mode is the shorthand
    ("sync": 1, the first pass only; "batch": 2, the first pass plus ONE batch retry round)."""
    cycles = batch_cycles if batch_cycles is not None else (2 if retry_mode == "batch" else 1)
    w = sla_h - processing_h * cycles - buffer_h
    if w <= 0:
        raise ValueError(f"SLA {sla_h} h cannot absorb {cycles} batch cycle(s) + {buffer_h} h buffer")
    return w


def batch_cycles_that_fit(sla_h: float, window_h: float, processing_h: float = 24, buffer_h: float = 2) -> int:
    """How many batch cycles the SLA can absorb for a given submission window; the pipeline caps batch rounds at this."""
    return max(0, math.floor((sla_h - window_h - buffer_h) / processing_h))


for sla, mode in [(30, "sync"), (36, "sync"), (54, "batch"), (30, "batch"), (24, "sync")]:
    try:
        print(f"SLA {sla:>2} h, retries via {mode:5}: submit every {submission_window(sla, retry_mode=mode):g} h")
    except ValueError as e:
        print(f"SLA {sla:>2} h, retries via {mode:5}: impossible ({e})")

# Simulate a week of arrivals with 4-hour windows and the worst-case processing time
rng = random.Random(3)
arrivals = sorted(rng.uniform(0, 7 * 24) for _ in range(2_000))
W, P, B = 4, 24, 2
ages = [(math.floor(t / W) + 1) * W - t + P + B for t in arrivals]
print(f"\n4 h windows: worst-case age {max(ages):.1f} h, median {sorted(ages)[len(ages) // 2]:.1f} h, "
      f"SLA breaches {sum(a > 30 for a in ages)}/{len(ages)}")
ages6 = [(math.floor(t / 6) + 1) * 6 - t + P + B for t in arrivals]
print(f"6 h windows: worst-case age {max(ages6):.1f} h, SLA breaches {sum(a > 30 for a in ages6)}/{len(ages6)}")

rounds = len(claude_pipe.batches)
allowed = claude_pipe.batch_cycles
print(f"\nThis run used {rounds} batch cycles ({[n for _, _, n in claude_pipe.batches]} requests), but retry_via='batch' with "
      f"max_rounds={claude_pipe.max_rounds} ALLOWS {allowed}: the budget must use the limit, not the typical run. "
      f"Worst case: {W} + 24 x {allowed} + {B} = {W + 24 * allowed + B} h.")
for sla in (30, 54, 102):
    k = batch_cycles_that_fit(sla, W)
    print(f"SLA {sla:>3} h with {W} h windows: {k} batch cycle(s) fit -> " +
          ("no batch at all" if k == 0 else "only the first pass via batch, retries synchronous" if k == 1
           else f"rounds 0-{k - 1} via batch, later rounds synchronous" if k < 4 else "every round may be a batch"))
sync_pipe = ExtractionPipeline(ClaudeBatchBackend((fs := FakeClaudeBatches()).client, sleep=fs.sleep), CORPUS, retry_via="sync")
_ = sync_pipe.run(verbose=False)
print("same documents, retries synchronous:", dict(coverage_report(sync_pipe.ledger)))
capped = ExtractionPipeline(ClaudeBatchBackend((fcap := FakeClaudeBatches()).client, sleep=fcap.sleep), CORPUS,
                            batch_cycles=batch_cycles_that_fit(54, W))
_ = capped.run(verbose=False)
print(f"54 h SLA, batch_cycles=2: batches {[n for _, _, n in capped.batches]}, round modes {capped.round_mode}")

> **Observe:** 6-hour windows plus the 2-hour buffer reach 32 hours and breach the SLA for a third of the documents; 4-hour windows meet it exactly, buffer included. The run in 3.5 needed two batch cycles, but its configuration allowed four (`max_rounds=3` means up to four submissions), and an SLA has to hold for what the configuration allows, not for a typical run. With a 30-hour SLA only the **first** pass can be a batch, so the retry rounds go synchronous (same outcomes, standard price for the small remainder). A 54-hour SLA absorbs exactly two cycles: `batch_cycles=2` sends the first retry round as a batch and any later round synchronously, so the worst case stays at 54 hours.

> **Try it:** the client asks for a 20-hour SLA on urgent suppliers. What do you recommend for those documents?

## 3.7 Confidence calibration and human review routing

The model reports a confidence per field. That number is a **ranking**, not a probability, until a **labeled validation set** shows what each band means for each field and document type. Calibrate thresholds per (document type, field), route only the fields below threshold (or with no safe threshold) to reviewers, prioritize by expected cost when capacity is short, and keep a **stratified random audit** of the auto-accepted fields. [Theory](README.md#37-confidence-calibration-and-human-review-routing)

In [ ]:
CALIBRATION = make_corpus([(lay, n * 6) for lay, n in PRODUCTION_PLAN], seed=99, prefix="cal", special=False)
REGISTRY.update({d.doc_id: d for d in CALIBRATION})

cal_backend = ClaudeBatchBackend((fcal := FakeClaudeBatches()).client, sleep=fcal.sleep)
bid = cal_backend.submit(first_pass_requests(CALIBRATION))
cal_backend.wait(bid)
records = []                                       # one row per (document, field): the calibration data
for o in cal_backend.results(bid):
    if o.status != "ok" or validate_extraction(o.data):
        continue
    d = REGISTRY[o.custom_id]
    for f in SCORED_FIELDS:
        records.append({"doc_type": o.data["doc_type"], "field": f, "confidence": o.data["field_confidence"][f],
                        "correct": field_correct(o.data, d.truth, f)})
print(f"{len(CALIBRATION)} labeled documents -> {len(records)} (document, field) records")


def reliability(rows: list[dict], edges=(0.0, 0.7, 0.8, 0.9, 0.95, 1.0)) -> list[tuple[str, int, float]]:
    out = []
    for i, (lo, hi) in enumerate(zip(edges, edges[1:])):
        last = i == len(edges) - 2
        hits = [r["correct"] for r in rows if lo <= r["confidence"] < hi or (last and r["confidence"] == hi)]
        if hits:
            out.append((f"{lo:.2f}-{hi:.2f}", len(hits), sum(hits) / len(hits)))
    return out


for dt in ("invoice", "receipt"):
    rows = [r for r in records if r["field"] == "vendor_name" and r["doc_type"] == dt]
    print(f"\nvendor_name on {dt}s: band, n, observed accuracy")
    for band, n, acc in reliability(rows):
        print(f"   {band}  n={n:4}  {acc:6.1%}")

> **Observe:** on invoices, a 0.9+ confidence for `vendor_name` really is ~100% accurate. On receipts the same 0.9 band is noticeably worse: OCR mistakes come back with high confidence. A single global threshold would auto-accept those errors. That is the calibration argument in one table.

In [ ]:
TARGET_ACCURACY, MIN_N = 0.97, 25


def pick_threshold(rows: list[dict], target: float = TARGET_ACCURACY, min_n: int = MIN_N) -> float | None:
    """Lowest cut-off whose auto-accepted set meets the target with at least min_n samples; None = always review."""
    for t in sorted({r["confidence"] for r in rows}):
        above = [r["correct"] for r in rows if r["confidence"] >= t]
        if len(above) < min_n:
            break
        if sum(above) / len(above) >= target:
            return t
    return None


THRESHOLDS: dict[tuple[str, str], float | None] = {}
for dt in sorted({r["doc_type"] for r in records}):
    for f in SCORED_FIELDS:
        THRESHOLDS[(dt, f)] = pick_threshold([r for r in records if r["doc_type"] == dt and r["field"] == f])
print(f"{'doc_type':15}" + "".join(f"{f[:10]:>11}" for f in SCORED_FIELDS))
for dt in sorted({k[0] for k in THRESHOLDS}):
    print(f"{dt:15}" + "".join(f"{('review' if THRESHOLDS[(dt, f)] is None else THRESHOLDS[(dt, f)]):>11}" for f in SCORED_FIELDS))

In [ ]:
FIELD_COST = {"stated_total_cents": 5, "due_date": 3, "vendor_name": 3, "document_number": 2, "issue_date": 2,
              "currency": 4, "payment_terms": 1, "doc_type": 2}      # relative cost of a wrong value reaching AP


def route(data: dict, thresholds: dict = THRESHOLDS) -> list[tuple[str, float]]:
    """Fields a reviewer must check, each with a priority = P(error) x cost. Empty list = auto-accept."""
    flagged = []
    for f in SCORED_FIELDS:
        t = thresholds.get((data["doc_type"], f))
        c = data["field_confidence"][f]
        if t is None or c < t:
            p_error = 1 - c if t is not None else max(0.05, 1 - c)
            flagged.append((f, round(p_error * FIELD_COST[f], 3)))
    return flagged


def stratified_audit(doc_ids: list[str], ledger: dict, rate: float = 0.10, min_per_type: int = 2, seed: int = 11) -> list[str]:
    """Random audit of AUTO-ACCEPTED documents, stratified by doc_type so small types are not drowned out."""
    rng = random.Random(seed)
    strata = defaultdict(list)
    for doc_id in doc_ids:
        strata[ledger[doc_id].data["doc_type"]].append(doc_id)
    sample = []
    for dt, ids in sorted(strata.items()):
        sample += rng.sample(ids, min(len(ids), max(min_per_type, round(rate * len(ids)))))
    return sample


WHOLE_DOC = len(SCORED_FIELDS)                      # a whole-document check costs as much reviewer time as checking every field
WHOLE_DOC_PRIORITY = sum(FIELD_COST.values())       # it blocks payment entirely, so it outranks any single flagged field


def build_review_plan(ledger: dict[str, LedgerEntry], capacity: int) -> dict:
    """One capacity-accounted queue for everything people must look at, in field-check units:
    1. the audit sample of auto-accepted documents (reserved first: it is the only unbiased error measurement),
    2. documents that ended in human_review (source conflicts, unresolved validation, refusals, exhausted attempts),
    3. flagged fields of accepted documents, by P(error) x cost."""
    tasks, auto = [], []
    for doc_id, e in ledger.items():
        if e.state == "human_review":
            tasks.append({"kind": "document", "doc_id": doc_id, "priority": WHOLE_DOC_PRIORITY, "units": WHOLE_DOC,
                          "reason": e.reason})
        elif fields := route(e.data):
            tasks += [{"kind": "field", "doc_id": doc_id, "field": f, "priority": prio, "units": 1} for f, prio in fields]
        else:
            auto.append(doc_id)
    audit_ids = stratified_audit(auto, ledger) if auto else []
    audit_tasks = [{"kind": "audit", "doc_id": d, "priority": float("inf"), "units": WHOLE_DOC} for d in audit_ids]
    today, deferred, used = [], [], 0
    for t in audit_tasks + sorted(tasks, key=lambda t: -t["priority"]):
        if t["kind"] == "audit" or used + t["units"] <= capacity:   # the audit is never deferred
            today.append(t)
            used += t["units"]
        else:
            deferred.append(t)
    return {"auto": auto, "audit": audit_ids, "today": today, "deferred": deferred, "used": used}


CAPACITY = 150                                      # field checks per day the AP team can absorb
plan = build_review_plan(claude_ledger, CAPACITY)
auto, audit = plan["auto"], plan["audit"]
print(f"accepted by validation: {sum(e.state == 'accepted' for e in claude_ledger.values())} | fully auto-accepted: {len(auto)} | "
      f"human_review documents: {sum(e.state == 'human_review' for e in claude_ledger.values())}")
print(f"today ({plan['used']}/{CAPACITY} field-check units):", dict(Counter(t["kind"] for t in plan["today"])))
print("deferred past today's capacity:", dict(Counter(t["kind"] for t in plan["deferred"])))
print("whole-document reviews and why:", dict(Counter(t["reason"].split(":")[0] for t in plan["today"] + plan["deferred"]
                                                       if t["kind"] == "document")))
print("audit sample by type:", dict(Counter(claude_ledger[i].data["doc_type"] for i in audit)))

> **Observe:** routing is per **field**, not per document: a reviewer checks two fields in seconds instead of re-keying an invoice. Everything people must look at shares **one** capacity budget: the audit sample is reserved first and never deferred (it is the only unbiased measurement of what the auto-accepted path gets wrong), then the documents that ended in `human_review` (source conflicts, unresolved failures, exhausted attempts), each costed as a full check, then the flagged fields by `P(error) x cost`. When the budget runs out, the low-priority field tail waits.

## 3.8 Accuracy by document type and field

Before anyone agrees to reduce review, break accuracy down by **document type × field** on labeled data. An aggregate number hides the weak segment. Here the production labels are available (synthetic data), so you can see exactly what the routing protects. [Theory](README.md#38-accuracy-by-document-type-and-field)

In [ ]:
def segment_accuracy(ledger: dict[str, LedgerEntry], docs: dict[str, Doc]) -> dict[tuple[str, str], tuple[int, int]]:
    seg = defaultdict(lambda: [0, 0])
    for doc_id, e in ledger.items():
        if e.data is None:
            continue
        truth = docs[doc_id].truth
        for f in SCORED_FIELDS:
            seg[(truth["doc_type"], f)][0] += field_correct(e.data, truth, f)
            seg[(truth["doc_type"], f)][1] += 1
    return {k: tuple(v) for k, v in seg.items()}


seg = segment_accuracy(claude_ledger, REGISTRY)
total_ok, total_n = sum(v[0] for v in seg.values()), sum(v[1] for v in seg.values())
print(f"aggregate field accuracy before review: {total_ok / total_n:.1%}  (n={total_n})\n")
print(f"{'doc_type':15}" + "".join(f"{f[:10]:>11}" for f in SCORED_FIELDS))
for dt in ["invoice", "receipt", "credit_note", "purchase_order", "other"]:
    cells = []
    for f in SCORED_FIELDS:
        ok, n = seg.get((dt, f), (0, 0))
        cells.append(f"{ok / n:.0%}{'*' if n and ok / n < TARGET_ACCURACY else ' '}".rjust(11) if n else "-".rjust(11))
    print(f"{dt:15}" + "".join(cells))
print("\n* below the 97% target")

auto_fields = [(d, f) for d in claude_ledger if claude_ledger[d].state == "accepted"
               for f in SCORED_FIELDS if f not in {x for x, _ in route(claude_ledger[d].data)}]
auto_ok = sum(field_correct(claude_ledger[d].data, REGISTRY[d].truth, f) for d, f in auto_fields)
print(f"accuracy of the fields that were AUTO-accepted after routing: {auto_ok / len(auto_fields):.1%} (n={len(auto_fields)})")

> **Observe:** the aggregate looks healthy, while the receipt row is well below target on `vendor_name` and `issue_date` (overconfident OCR errors), and invoices miss the target on `document_number` and `due_date` (transposed digits in letters, a slipped day in tables). The fields that routing auto-accepts are far more accurate than the aggregate, because the calibrated thresholds sent exactly the weak segments to people. This is the evidence a client needs before reducing human review, and it has to be re-measured whenever the model, prompt or document mix changes.

## 3.9 Cost comparison

Price the run from the token usage the pipeline recorded, at the dated list prices below, and add the human-review cost: the number that matters is **cost per accepted document**, not cost per token. (Pricing arithmetic is out of scope for the CCAR-F exam; it is very much in scope for the job.) [Theory](README.md#39-cost-comparison)

In [ ]:
# USD per 1M tokens (standard input, output). Batch = 50% of standard on both vendors. Checked 2026-10-03.
PRICES = {"claude-sonnet-5-5": (2.00, 10.00), "claude-haiku-4-5": (1.00, 5.00),
          "gpt-6-luna": (0.10, 0.50), "gpt-6.1-sol": (2.00, 10.00)}
REVIEW_COST_PER_FIELD = 0.40                       # assumption: ~1 minute of an AP clerk per flagged field


def run_cost(pipe: ExtractionPipeline, model: str, retry_mode: str | None = None) -> dict:
    """retry_mode=None prices each retry round as it actually ran (pipe.round_mode); "batch"/"sync" prices all retries that way."""
    p_in, p_out = PRICES[model]
    first = pipe.usage[0][0] * p_in * 0.5 / 1e6 + pipe.usage[0][1] * p_out * 0.5 / 1e6
    def mult(r: int) -> float:
        return 0.5 if (retry_mode or pipe.round_mode.get(r, "batch")) == "batch" else 1.0
    retries = sum((u[0] * p_in + u[1] * p_out) * mult(r) for r, u in pipe.usage.items() if r > 0) / 1e6
    accepted = [e for e in pipe.ledger.values() if e.state == "accepted"]
    field_reviews = sum(len(route(e.data)) for e in accepted) + sum(len(SCORED_FIELDS) for e in pipe.ledger.values()
                                                                     if e.state == "human_review")
    human = field_reviews * REVIEW_COST_PER_FIELD
    total = first + retries + human
    return {"model": model, "first_pass": first, "retries": retries, "human": human, "field_reviews": field_reviews,
            "per_doc": total / len(pipe.ledger),                       # for volume projections
            "per_accepted": total / max(1, len(accepted))}             # the budgeting number: cost per ACCEPTED document


rows = [run_cost(claude_pipe, "claude-sonnet-5-5"), run_cost(claude_pipe, "claude-haiku-4-5"),
        run_cost(openai_pipe, "gpt-6-luna"), run_cost(openai_pipe, "gpt-6.1-sol"),
        run_cost(claude_pipe, "claude-sonnet-5-5", retry_mode="sync")]
print(f"{'model':34} {'batch pass':>10} {'retries':>9} {'model $/1k docs':>16} {'human':>8} {'per doc':>8} {'per accepted doc':>17}")
for r, label in zip(rows, ["", "", "", "", " (sync retries)"]):
    model_per_1k = (r["first_pass"] + r["retries"]) / len(claude_pipe.ledger) * 1000
    print(f"{r['model'] + label:34} ${r['first_pass']:9.4f} ${r['retries']:8.4f} ${model_per_1k:15.2f} ${r['human']:7.2f} "
          f"${r['per_doc']:7.4f} ${r['per_accepted']:16.4f}")
tokens = claude_pipe.usage
print(f"\ntokens (simulated, ~4 chars/token): first pass in/out = {tokens[0][0]:,}/{tokens[0][1]:,}; "
      f"retries in/out = {sum(u[0] for r, u in tokens.items() if r):,}/{sum(u[1] for r, u in tokens.items() if r):,}")
monthly = 100_000
print(f"projection at {monthly:,} documents/month (same mix): " +
      ", ".join(f"{r['model']} ≈ ${r['per_doc'] * monthly:,.0f}" for r in rows[:4]))

> **Observe:** at these volumes the model bill is cents; **human review dominates** cost per document. "Per accepted doc" divides everything (tokens, retries, every field review, including the documents sent wholly to people) by the documents the pipeline accepted, so it is the conservative number to put in a budget; "per doc" is what you multiply by monthly volume. That changes the optimization target: a cheaper model that sends 10% more fields to reviewers costs more overall. The model comparison here reuses the same simulated quality for every model, which is exactly what you must **not** assume in real life: run the labeled sample on each candidate model and recompute both the token cost and the review rate before choosing.

> **Try it:** set `REVIEW_COST_PER_FIELD = 0.10` and find the break-even review-rate difference between `gpt-6-luna` and `claude-sonnet-5-5`.

## Mini-project: the end-to-end pipeline on both stacks

One function runs the whole thing for a vendor: pre-flight (optional), batch first pass, failures resubmitted by `custom_id` (synchronously when the SLA has no room for another cycle), validation-retry with feedback, calibrated routing, the audit sample, accuracy by segment, and a cost line. Then the two vendors side by side.

In [ ]:
def run_end_to_end(vendor: Literal["claude", "openai"], docs: list[Doc], *, preflight: bool, sla_h: float = 30,
                   live_clients: dict | None = None) -> dict:
    if live_clients:
        client = live_clients[vendor]
        backend = ClaudeBatchBackend(client) if vendor == "claude" else OpenAIBatchBackend(client)
    else:
        fake = FakeClaudeBatches() if vendor == "claude" else FakeOpenAIBatches()
        backend = (ClaudeBatchBackend if vendor == "claude" else OpenAIBatchBackend)(fake.client, sleep=fake.sleep)
    cycles = batch_cycles_that_fit(sla_h, window_h=4)  # 4-hour submission windows, as in 3.6
    if cycles < 1:
        raise ValueError(f"a {sla_h} h SLA cannot absorb even one batch cycle: process these documents synchronously")
    pipe = ExtractionPipeline(backend, docs, batch_cycles=cycles, preflight=preflight)   # later rounds go synchronous
    retry_via = "sync" if cycles == 1 else f"batch x{min(cycles, pipe.max_rounds + 1) - 1}, then sync"
    ledger = pipe.run(verbose=False)
    accepted = [d for d, e in ledger.items() if e.state == "accepted"]
    auto = [d for d in accepted if not route(ledger[d].data)]
    seg = segment_accuracy(ledger, REGISTRY)
    model = CLAUDE_MODEL if vendor == "claude" else OPENAI_MODEL
    cost = run_cost(pipe, model if model in PRICES else ("claude-sonnet-5-5" if vendor == "claude" else "gpt-6-luna"))
    return {"vendor": vendor, "preflight": preflight, "retry_via": retry_via, "batches": [n for _, _, n in pipe.batches],
            "states": coverage_report(ledger), "auto_accepted": len(auto), "audit": len(stratified_audit(auto, ledger)) if auto else 0,
            "field_reviews": cost["field_reviews"], "accuracy": sum(v[0] for v in seg.values()) / max(1, sum(v[1] for v in seg.values())),
            "cost_per_accepted": cost["per_accepted"], "ledger": ledger}


runs = [run_end_to_end(v, CORPUS, preflight=p) for v in ("claude", "openai") for p in (False, True)]
for r in runs:
    terminal = {k.replace("accepted ", "acc "): v for k, v in sorted(r["states"].items())}
    print(f"{r['vendor']:6} preflight={str(r['preflight']):5} retries={r['retry_via']:5} batch sizes={r['batches']} "
          f"auto={r['auto_accepted']:2} audit={r['audit']} reviews={r['field_reviews']:3} acc={r['accuracy']:.1%} "
          f"${r['cost_per_accepted']:.3f}/accepted doc")
    print("        ", terminal)

> **Observe:** with a 30-hour SLA the retries run synchronously on both stacks, so each run has exactly one batch cycle. Pre-flight token counting removes the "prompt too long" round trip: the oversized documents go in as chunks on the first pass (112 requests instead of 100) and are accepted there, instead of waiting for a resubmission. Both vendors reach the same terminal states on the same inputs; the decision between them rests on measured quality per segment, price, data-governance terms and platform fit, not on the batch mechanics.

### Live run (opt-in): the same code against the real batch APIs

Set `RUN_LIVE = True` in the setup cell (and add both keys to `.env`) to send `LIVE_SAMPLE` documents to each real batch API with the default models. Real batches usually finish within an hour but may take up to 24 hours; the wait gives up after `max_wait_s`, and re-running the cell later re-attaches to the saved batch IDs instead of submitting new batches (results are matched by `custom_id`). The model's real answers are scored against the synthetic labels.

In [ ]:
if not RUN_LIVE:
    print("RUN_LIVE is False: skipping the live run (nothing was sent).")
else:
    live = {"claude": anthropic.Anthropic(), "openai": OpenAI()}
    sample = [d for d in CORPUS if not d.tags][:LIVE_SAMPLE]           # clean documents only: no fake-only failures
    # Batch IDs survive a re-run and a kernel restart: a re-run re-attaches instead of paying for a second batch.
    LIVE_STATE = Path(tempfile.gettempdir()) / "cap03_live_batches.json"   # outside the repo
    saved = json.loads(LIVE_STATE.read_text()) if LIVE_STATE.exists() else {}
    for vendor in ("claude", "openai"):
        backend = (ClaudeBatchBackend(live["claude"], max_wait_s=3600) if vendor == "claude"
                   else OpenAIBatchBackend(live["openai"], max_wait_s=3600))
        if vendor in saved:
            bid = saved[vendor]
            print(vendor, "re-attaching to", bid, "(nothing resubmitted)")
        else:
            bid = saved[vendor] = backend.submit(first_pass_requests(sample))
            LIVE_STATE.write_text(json.dumps(saved))
        status = backend.wait(bid)
        print(vendor, "batch", bid, "->", status)
        if status == "timeout":                                        # still running: do NOT read results yet
            print(f"   not finished yet; re-run this cell later (the ID is saved in {LIVE_STATE})")
            continue
        outcomes = backend.results(bid)
        del saved[vendor]
        LIVE_STATE.write_text(json.dumps(saved))
        for o in outcomes:
            if o.status == "ok":
                acc = sum(field_correct(o.data, REGISTRY[o.custom_id].truth, f) for f in SCORED_FIELDS) / len(SCORED_FIELDS)
                print(f"   {o.custom_id}: ok, field accuracy vs label {acc:.0%}, issues {[i.code for i in validate_extraction(o.data)]}")
            else:
                print(f"   {o.custom_id}: {o.status} {o.error_type or ''}")

## Recap

- **Schema:** one canonical contract, two strict wire schemas. Nullable for absent values, enum + `"other"` + detail for open categories, `"unclear"` for vague ones; Claude allows truly optional keys, OpenAI strict needs required + nullable. A naive schema turns gaps into fabrications that no validator can see.
- **Prompt:** few-shot examples for the hard layouts, chosen and measured on a labeled sample **before** the big run.
- **Validation and retry:** specific field-level errors fed back with the document and the failed extraction fix format and structure; absent information stays `null` and goes to a person after one try. Never pressure the model into a value.
- **Batch:** `custom_id` per document (`-cN` per chunk); resubmit only failures, each with the right modification; reconcile so every document ends in one terminal state; persist the full resumable state so a crash or a wait timeout re-attaches to the batch instead of resubmitting it; never read results before a terminal status; catch synchronous errors per request. Same design on the Message Batches API and the Batch API; different plumbing.
- **SLA:** `W + 24 h x batch cycles + B ≤ SLA`, counted for the cycles the configuration **allows**, not the typical run; cap batch rounds at what the SLA absorbs (`batch_cycles`) and run later rounds synchronously.
- **Calibration and routing:** thresholds per document type × field from labeled data; route fields, prioritize by expected cost, always keep the stratified audit.
- **Cost:** report cost per accepted document including review, not cost per token.

Next: [homework](02_homework.ipynb)